In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import time


In [ ]:
# --- Check for GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### Agent brain (unchanged architecture from the gaze-cooperation model)\nOnly the *input size* changes (public input grows from 5 to 7 to carry self/partner effort cost). Hidden size, heads, and everything else stays identical, which is what makes the weight-transfer loader below possible.

In [ ]:
#################################################################
# 1. AGENT BRAIN CLASS (unchanged from gaze-cooperation model)
#################################################################

class AgentBrain(nn.Module):
    """
    An Actor-Critic agent with a shared RNN trunk (GRU).
    Identical to the gaze-cooperation model -- only input_size changes
    at construction time (7 public + 5 private = 12, vs. 5+5=10 before).
    """
    def __init__(self, public_input_size, private_input_size, rnn_hidden_size,
                 move_action_size, pull_action_size, gaze_action_size):

        super(AgentBrain, self).__init__()

        self.input_size = public_input_size + private_input_size
        self.rnn_hidden_size = rnn_hidden_size
        self.move_action_size = move_action_size
        self.pull_action_size = pull_action_size
        self.gaze_action_size = gaze_action_size

        # --- 1. The Shared RNN "Trunk" ---
        self.rnn_trunk = nn.GRUCell(self.input_size, self.rnn_hidden_size)

        # --- 2. The "Actor" Head ---
        self.move_head = nn.Linear(self.rnn_hidden_size, move_action_size)
        self.pull_head = nn.Linear(self.rnn_hidden_size, pull_action_size)
        self.gaze_head = nn.Linear(self.rnn_hidden_size, gaze_action_size)

        # --- 3. The "Critic" Head ---
        self.critic_head = nn.Linear(self.rnn_hidden_size, 1)

    def forward(self, public_input, private_gaze_input, prev_hidden_state, pull_mask):

        full_input = torch.cat([public_input, private_gaze_input], dim=-1)
        current_hidden_state = self.rnn_trunk(full_input, prev_hidden_state)

        value = self.critic_head(current_hidden_state)

        move_logits = self.move_head(current_hidden_state)
        gaze_logits = self.gaze_head(current_hidden_state)

        pull_logits = self.pull_head(current_hidden_state)
        pull_logits = pull_logits + pull_mask

        return (move_logits, pull_logits, gaze_logits), value, current_hidden_state


### environment: effort-inequity cooperation task (RANDOMIZED pull cooldown)
Same single-factor effort design as the fixed-cooldown version, PLUS randomized pull cooldown (matching your gaze-cooperation `COOP_FULL_RANDOM` phase): instead of a fixed `pull_reset_time=20`, cooldown after each pull is drawn uniformly from `[cool_min, cool_max]`. This removes the fully-predictable timing cue agents could exploit in the fixed-cooldown version -- a robustness check on top of whatever effort-sensitivity result the fixed-cooldown model shows, not a replacement for it.

- `cool_min=20, cool_max=20` by default (identical to old fixed behavior) for phases other than `"COOP_FULL_RANDOM"`
- `cool_min=20, cool_max=60` when phase is explicitly set to `"COOP_FULL_RANDOM"` -- matches your original randomized-cooldown gaze notebook
- Everything else (force/session/block design, force-to-cost mapping) is unchanged from the fixed-cooldown effort notebook

In [ ]:
#################################################################
# 2. EFFORT-INEQUITY ENVIRONMENT (single-factor pilot design, session-structured,
#    RANDOMIZED pull cooldown)
#################################################################

# --- Force -> RL cost mapping (unchanged from fixed-cooldown version) ---
FORCE_LOW = 100          # grams -- the fixed agent's constant force
VARYING_FORCE_RANGE = (100, 900)   # grams -- the varying agent's per-block range
FORCE_TO_COST_SCALE = 0.3 / 900    # so 900g ~ comparable magnitude to old EFFORT_HIGH=0.3
SESSION_BLOCKS = 4        # blocks per "session" -- matches K99 text (4 blocks/session)

def force_to_cost(force_grams):
    return force_grams * FORCE_TO_COST_SCALE


class CurriculumEnv:
    def __init__(self, block_len=3000, session_blocks=SESSION_BLOCKS, lock_fixed_agent=None):
        # lock_fixed_agent: 'A' or 'B' locks that agent as ALWAYS fixed (skips the
        # session-flip). None (default) preserves original behavior -- fixed_agent
        # re-randomizes every `session_blocks` blocks. Backward compatible: any
        # existing CurriculumEnv(...) call without this arg is unaffected.
        self.training_phase = "SELF_REWARD"
        self.current_sync_window = 30

        # --- Cooldown: randomized when phase == "COOP_FULL_RANDOM", fixed otherwise ---
        self.cool_min = 20
        self.cool_max = 20   # cool_min==cool_max => identical to old fixed pull_reset_time=20

        # --- Gaze cost (kept from the gaze-cooperation model) ---
        self.gaze_cost = 0.0

        # --- Effort-inequity: single-factor design, session-structured ---
        self.block_len = block_len
        self.session_blocks = session_blocks
        self._effort_step_counter = 0
        self._block_counter = 0
        self.fixed_agent = None  # assigned on the first block below
        self.lock_fixed_agent = lock_fixed_agent

        self.reset()
        self._set_new_block()

    def set_training_phase(self, phase, sync_window=30):
        self.training_phase = phase
        self.current_sync_window = sync_window
        if phase == "COOP_FULL_RANDOM":
            self.cool_min, self.cool_max = 20, 60
        else:
            self.cool_min, self.cool_max = 20, 20
        print(f"*** ENV UPDATED: Phase={self.training_phase}, "
              f"SyncWindow={self.current_sync_window}, GazeCost={self.gaze_cost}, "
              f"Cooldown=[{self.cool_min},{self.cool_max}] ***")

    def _sample_cooldown(self):
        return np.random.randint(self.cool_min, self.cool_max + 1)

    def _set_new_block(self):
        """
        Advances one block. At the start of every new session (every
        `session_blocks` blocks), re-randomizes which agent is fixed --
        mimicking "the animal with the fixed force will be randomly
        alternated session by session." Within a session, the fixed agent
        stays fixed at FORCE_LOW while the varying agent's force is
        resampled uniformly from VARYING_FORCE_RANGE each block.

        If lock_fixed_agent is set, skips the session-flip entirely and
        keeps that agent fixed for the whole run.
        """
        if self.lock_fixed_agent is not None:
            self.fixed_agent = self.lock_fixed_agent
        elif self._block_counter % self.session_blocks == 0:
            self.fixed_agent = np.random.choice(['A', 'B'])

        varying_force = np.random.uniform(*VARYING_FORCE_RANGE)
        if self.fixed_agent == 'A':
            self.force_A, self.force_B = FORCE_LOW, varying_force
        else:
            self.force_A, self.force_B = varying_force, FORCE_LOW

        self.cost_A = force_to_cost(self.force_A)
        self.cost_B = force_to_cost(self.force_B)
        self.cond_label = f"fixed{self.fixed_agent}_A{self.force_A:.0f}g_B{self.force_B:.0f}g"

        self._block_counter += 1

    def _get_state(self):
        on_cooldown_A = self.agent_A_pull_cooldown > 0
        is_drinking_A = self.A_is_drinking
        can_pull_A = not (on_cooldown_A or is_drinking_A)

        on_cooldown_B = self.agent_B_pull_cooldown > 0
        is_drinking_B = self.B_is_drinking
        can_pull_B = not (on_cooldown_B or is_drinking_B)

        state_A = {
            'can_pull': can_pull_A,
            'self_is_drinking': self.A_is_drinking,
            'self_on_cooldown': on_cooldown_A,
            'self_cooldown_remaining': self.agent_A_pull_cooldown,
            'self_drink_timer_remaining': self.A_drink_timer,
            'success_sound': self.success_sound_played
        }
        state_B = {
            'can_pull': can_pull_B,
            'self_is_drinking': self.B_is_drinking,
            'self_on_cooldown': on_cooldown_B,
            'self_cooldown_remaining': self.agent_B_pull_cooldown,
            'self_drink_timer_remaining': self.B_drink_timer,
            'success_sound': self.success_sound_played
        }
        return state_A, state_B

    def reset(self):
        self.current_step = 0
        self.pull_A_timestamp = -1
        self.pull_B_timestamp = -1
        self.agent_A_pull_cooldown = 0
        self.agent_B_pull_cooldown = 0
        self.success_sound_played = 0
        self.A_time_since_pull = 1000
        self.B_time_since_pull = 1000
        self.A_is_drinking = False
        self.B_is_drinking = False
        self.A_drink_timer = 0
        self.B_drink_timer = 0
        return self._get_state()

    def step(self, action_A, action_B):
        self.current_step += 1
        reward_A, reward_B = -0.01, -0.01  # base cost of living
        self.success_sound_played = 0
        done = False
        coordination_attempt_complete = False

        self.A_time_since_pull += 1
        self.B_time_since_pull += 1

        # --- Gaze cost (unchanged from gaze-cooperation model) ---
        if self.training_phase != "SELF_REWARD":
            if action_A['gaze'] == 1:
                reward_A -= self.gaze_cost
            if action_B['gaze'] == 1:
                reward_B -= self.gaze_cost

        # --- Standard timer logic ---
        on_cooldown_A = self.agent_A_pull_cooldown > 0
        is_drinking_A = self.A_is_drinking
        can_pull_A = not (on_cooldown_A or is_drinking_A)

        on_cooldown_B = self.agent_B_pull_cooldown > 0
        is_drinking_B = self.B_is_drinking
        can_pull_B = not (on_cooldown_B or is_drinking_B)

        if self.A_drink_timer > 0:
            self.A_drink_timer -= 1
            if self.A_drink_timer == 0: self.A_is_drinking = False
        if self.B_drink_timer > 0:
            self.B_drink_timer -= 1
            if self.B_drink_timer == 0: self.B_is_drinking = False
        if self.agent_A_pull_cooldown > 0: self.agent_A_pull_cooldown -= 1
        if self.agent_B_pull_cooldown > 0: self.agent_B_pull_cooldown -= 1

        # --- Effort cost on ATTEMPTED pulls (mask-permitting) ---
        if action_A['pull'] == 1 and can_pull_A:
            reward_A -= self.cost_A
        if action_B['pull'] == 1 and can_pull_B:
            reward_B -= self.cost_B

        if self.training_phase == "SELF_REWARD":
            if (action_A['pull'] == 1 and can_pull_A):
                self.A_time_since_pull = 0; reward_A += 15.0; self.success_sound_played = 1
                self.agent_A_pull_cooldown = self._sample_cooldown()
                self.A_is_drinking = True; self.A_drink_timer = np.random.randint(30, 50)
            if (action_B['pull'] == 1 and can_pull_B):
                self.B_time_since_pull = 0; reward_B += 15.0; self.success_sound_played = 1
                self.agent_B_pull_cooldown = self._sample_cooldown()
                self.B_is_drinking = True; self.B_drink_timer = np.random.randint(30, 50)

        else:  # COOP_ASSISTED, COOP_FULL, or COOP_FULL_RANDOM
            if (action_A['pull'] == 1 and can_pull_A):
                self.A_time_since_pull = 0
                self.pull_A_timestamp = self.current_step
                self.agent_A_pull_cooldown = self._sample_cooldown()
                if self.pull_B_timestamp != -1: coordination_attempt_complete = True

            if (action_B['pull'] == 1 and can_pull_B):
                self.B_time_since_pull = 0
                self.pull_B_timestamp = self.current_step
                self.agent_B_pull_cooldown = self._sample_cooldown()
                if self.pull_A_timestamp != -1: coordination_attempt_complete = True

            if coordination_attempt_complete:
                if abs(self.pull_A_timestamp - self.pull_B_timestamp) <= self.current_sync_window:
                    reward_A += 15.0; reward_B += 15.0
                    self.success_sound_played = 1
                    self.A_is_drinking = True; self.A_drink_timer = np.random.randint(30, 50)
                    self.B_is_drinking = True; self.B_drink_timer = np.random.randint(30, 50)
                else:
                    if self.training_phase == "COOP_ASSISTED": reward_A += 5.0; reward_B += 5.0
                    else: reward_A -= 0.5; reward_B -= 0.5
                self.pull_A_timestamp = -1
                self.pull_B_timestamp = -1

        # --- Advance effort block (and, every session_blocks blocks, re-flip fixed_agent) ---
        self._effort_step_counter += 1
        if self._effort_step_counter % self.block_len == 0:
            self._set_new_block()

        next_state_A, next_state_B = self._get_state()
        return next_state_A, next_state_B, reward_A, reward_B, done

### helper functions\n`get_private_state`, `calculate_gae`, `sample_actions` are unchanged from the gaze-cooperation model. `preprocess_state` is replaced with `preprocess_state_with_effort`, which appends normalized self/partner effort cost to the public vector.

In [ ]:
#################################################################
# 3. HELPER FUNCTIONS
#################################################################

def preprocess_state_with_effort(state_dict, self_force, partner_force):
    """
    Public input, now size 7: the original 5 dims + normalized self_force + partner_force.

    IMPORTANT: this takes FORCE (grams), not cost, and normalizes it directly
    to [0, 1] via (force - FORCE_LOW) / (VARYING_FORCE_RANGE[1] - FORCE_LOW).
    The fixed agent (always FORCE_LOW=100g) gets exactly 0.0; the varying
    agent ranges continuously from 0.0 (at 100g) to 1.0 (at 900g).

    BUG HISTORY: earlier version passed COST and binarized it
    (`1.0 if cost > 0 else 0.0`) -- that made sense for the old discrete
    EFFORT_LOW=0.0/EFFORT_HIGH=0.3 design, but broke silently once
    FORCE_LOW became 100g (never zero): self_cost was ALWAYS > 0, so the
    network received a constant 1.0 regardless of actual force, and had
    NO information to condition behavior on. This fix restores that signal.
    """
    is_drinking = torch.tensor([1.0 if state_dict['self_is_drinking'] else 0.0])
    can_pull = torch.tensor([1.0 if state_dict['can_pull'] else 0.0])
    success_sound = torch.tensor([1.0 if state_dict['success_sound'] else 0.0])
    drink_timer = torch.tensor([state_dict['self_drink_timer_remaining'] / 50.0])
    cooldown_timer = torch.tensor([state_dict['self_cooldown_remaining'] / 20.0])

    force_range = VARYING_FORCE_RANGE[1] - FORCE_LOW  # = 800
    self_force_norm = torch.tensor([np.clip((self_force - FORCE_LOW) / force_range, 0.0, 1.0)])
    partner_force_norm = torch.tensor([np.clip((partner_force - FORCE_LOW) / force_range, 0.0, 1.0)])

    tensor_state = torch.cat([
        is_drinking, can_pull, success_sound, drink_timer, cooldown_timer,
        self_force_norm, partner_force_norm
    ], dim=0)

    return tensor_state.float().unsqueeze(0).to(device)


def get_private_state(partner_state_dict, partner_time, sync_window, reliability_level, is_gazing):
    """
    Unchanged from the gaze-cooperation model. Gaze-gated noisy readout of
    partner state. Reliability is held FIXED (not swept) during effort
    training -- see hyperparameters cell -- to avoid conflating two sources
    of task difficulty at once.
    """
    if partner_state_dict['can_pull']:
        true_mean = 1.0
    else:
        true_mean = 0.0

    base_noise = 0.6
    current_sigma = base_noise / (reliability_level + 1e-2)

    if is_gazing:
        observed_value = np.random.normal(loc=true_mean, scale=current_sigma)
        valid_bit = 1.0
    else:
        observed_value = 0.0
        valid_bit = 0.0

    p_signal = torch.tensor([observed_value])
    p_time = torch.tensor([min(partner_time / sync_window, 1.0)])
    p_drink = torch.tensor([1.0 if partner_state_dict['self_is_drinking'] else 0.0])
    p_cool = torch.tensor([1.0 if partner_state_dict['self_on_cooldown'] else 0.0])
    v_bit = torch.tensor([valid_bit])

    tensor_state = torch.cat([p_signal, p_time, p_drink, p_cool, v_bit], dim=0)
    return tensor_state.float().unsqueeze(0).to(device)


def calculate_gae(rewards, values, next_value, gamma=0.99, lambda_=0.95):
    gae = 0
    returns = []
    values = values + [next_value]
    for step in reversed(range(len(rewards))):
        delta = rewards[step] + gamma * values[step + 1] - values[step]
        gae = delta + gamma * lambda_ * gae
        returns.insert(0, gae + values[step])
    return torch.cat(returns).view(-1).float().to(device)


def sample_actions(logits_tuple):
    move_logits, pull_logits, gaze_logits = logits_tuple

    move_dist = Categorical(logits=move_logits)
    move_action = move_dist.sample()
    move_log_prob = move_dist.log_prob(move_action)

    pull_dist = Categorical(logits=pull_logits)
    pull_action = pull_dist.sample()
    pull_log_prob = pull_dist.log_prob(pull_action)

    gaze_dist = Categorical(logits=gaze_logits)
    gaze_action = gaze_dist.sample()
    gaze_log_prob = gaze_dist.log_prob(gaze_action)

    total_log_prob = move_log_prob + pull_log_prob + gaze_log_prob

    actions_dict = {
        'move': move_action.item(),
        'pull': pull_action.item(),
        'gaze': gaze_action.item()
    }
    return actions_dict, total_log_prob

### weight transfer: load the gaze-cooperation generalist checkpoint into the wider effort model\nOnly `rnn_trunk.weight_ih` changes shape (input size 10 -> 12). Everything else (recurrent weights, biases, all four output heads) transfers directly since those only depend on `hidden_size`, which is unchanged.\n\n**Path note:** this notebook lives in `MARL_effort_based_cooperation/`, while the source checkpoints live in `MARL_gaze_accumulation_strategies/training_withGAE_PPO_randomPullCoolTime/`. The path below is a **relative** reference between those two sibling folders -- adjust `OLD_CHECKPOINT_DIR` if your actual folder depth differs, or point it at a local copy if you'd rather not depend on the sibling folder (safer for sbatch jobs, since working directory at submission time isn't always what you expect).

In [ ]:
#################################################################
# 4. WEIGHT TRANSFER
#################################################################

OLD_CHECKPOINT_DIR = "/gpfs/radev/home/ws523/marmoset_tracking_DLCv2/following_up_analysis/"+\
"MARL_gaze_accumulation_strategies/training_withGAE_PPO_randomPullCoolTime/"

OLD_PUBLIC_INPUT_SIZE = 5
OLD_PRIVATE_INPUT_SIZE = 5

def load_and_expand_agent(new_model, old_checkpoint_path,
                           old_public_size=OLD_PUBLIC_INPUT_SIZE,
                           old_private_size=OLD_PRIVATE_INPUT_SIZE,
                           new_public_size=7,
                           new_private_size=5):
    old_state = torch.load(old_checkpoint_path, map_location='cpu')
    new_state = new_model.state_dict()

    for key, old_tensor in old_state.items():
        if key == 'rnn_trunk.weight_ih':
            new_tensor = new_state[key].clone()  # random init for new columns
            new_tensor[:, :old_public_size] = old_tensor[:, :old_public_size]
            new_tensor[:, new_public_size:new_public_size + old_private_size] = \
                old_tensor[:, old_public_size:old_public_size + old_private_size]
            new_state[key] = new_tensor
            print(f"Expanded '{key}': {tuple(old_tensor.shape)} -> {tuple(new_tensor.shape)} "
                  f"(effort-cost columns randomly initialized)")
        else:
            new_state[key] = old_tensor

    new_model.load_state_dict(new_state)
    print(f"Loaded and expanded weights from {old_checkpoint_path}")
    return new_model


### training hyperparameters

In [ ]:
#################################################################
# 5. TRAINING HYPERPARAMETERS
#################################################################

RNN_HIDDEN_SIZE = 128
PUBLIC_INPUT_SIZE = 7    # 5 original + self_cost + partner_cost
PRIVATE_INPUT_SIZE = 5   # unchanged

MOVE_ACTIONS = 5
PULL_ACTIONS = 2
GAZE_ACTIONS = 2

TARGET_WINDOW = 10        # keep the "1s" sync window from the trained base
FIXED_RELIABILITY = 1.0   # held fixed during effort training -- see note above
BLOCK_LEN = 3000          # steps per effort contingency block (~6 rollouts)

LR_FINE_TUNE = 5e-5       # matches your other continue_training_* fine-tunes
GAMMA = 0.99
N_STEPS = 500
CONTINUE_STEPS = 800000   # starting point -- extend if behavior hasn't stabilized
ENTROPY_COEF = 0.0001
VALUE_LOSS_COEF = 0.1
EPS_CLIP = 0.2

GAZE_COST = 0.2   # carried over from the efficient/generalist gaze model


### run the training with different sensitivity to the effort level to figure out the best choice

In [ ]:
"""
Cost-scale sweep: runs several SHORT fine-tunes (not the full 800k) at
different FORCE_TO_COST_SCALE multipliers, to find the smallest scale where
attempt rate actually starts responding to own cost -- before spending a
full training run on a scale that might still be too weak.

Requires: AgentBrain, CurriculumEnv, preprocess_state_with_effort,
get_private_state, calculate_gae, sample_actions, load_and_expand_agent,
and the hyperparameter constants already defined earlier in the notebook.
Paste this in a new cell after the main training-loop cell.
"""

import copy

SWEEP_STEPS = 100000          # short probe run, not the full 800k
SWEEP_SCALES = [1.0, 3.0, 5.0, 8.0, 12.0, 16.0, 20.0, 25.0]   # wider range this time
BASE_SCALE = 0.3 / 900          # current value, for reference
CHECKPOINT_SET = "generalist_withGAE_PPO"  # switched from "generalist_withGAE_PPO" or "efficient_expert"


def run_short_probe(cost_scale_multiplier, n_steps=SWEEP_STEPS, seed=None,  entropy_coef=None):
    """
    Runs a short fine-tune at a given cost-scale multiplier and returns
    the correlation between own force and own attempt rate, plus overall
    conditional attempt rate -- both computed only over ELIGIBLE steps.

    seed: if given, sets np.random and torch seeds at the start so this
    specific replicate is reproducible. Different seeds give independent
    replicates of the same multiplier, for checking whether an effect is
    real or just run-to-run noise.
    """
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    entropy_coef_used = entropy_coef if entropy_coef is not None else ENTROPY_COEF
    
    global FORCE_TO_COST_SCALE
    FORCE_TO_COST_SCALE = BASE_SCALE * cost_scale_multiplier

    env = CurriculumEnv(block_len=BLOCK_LEN)
    env.set_training_phase("COOP_FULL_RANDOM", sync_window=TARGET_WINDOW)
    env.gaze_cost = GAZE_COST

    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_A = load_and_expand_agent(agent_A, f"{OLD_CHECKPOINT_DIR}/agent_A_{CHECKPOINT_SET}.pth")
    agent_B = load_and_expand_agent(agent_B, f"{OLD_CHECKPOINT_DIR}/agent_B_{CHECKPOINT_SET}.pth")

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state_with_effort(state_A_dict, env.force_A, env.force_B)
    public_B = preprocess_state_with_effort(state_B_dict, env.force_B, env.force_A)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)

    log = []

    for update_step in range(n_steps // N_STEPS):
        mem_pub_A, mem_priv_A, mem_mask_A = [], [], []
        mem_act_A, mem_val_A, mem_rew_A, mem_logp_A = [], [], [], []
        mem_pub_B, mem_priv_B, mem_mask_B = [], [], []
        mem_act_B, mem_val_B, mem_rew_B, mem_logp_B = [], [], [], []

        start_hidden_A, start_hidden_B = hidden_A.detach(), hidden_B.detach()
        hidden_A, hidden_B = start_hidden_A, start_hidden_B

        for _ in range(N_STEPS):
            mask_A = torch.tensor([[0., 0. if state_A_dict['can_pull'] else -1e9]], device=device).float()
            mask_B = torch.tensor([[0., 0. if state_B_dict['can_pull'] else -1e9]], device=device).float()

            mem_pub_A.append(public_A); mem_priv_A.append(private_A); mem_mask_A.append(mask_A)
            mem_pub_B.append(public_B); mem_priv_B.append(private_B); mem_mask_B.append(mask_B)

            with torch.no_grad():
                logits_A, value_A, hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
                logits_B, value_B, hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)

            act_A, log_prob_A = sample_actions(logits_A)
            act_B, log_prob_B = sample_actions(logits_B)
            mem_act_A.append(act_A); mem_val_A.append(value_A); mem_logp_A.append(log_prob_A)
            mem_act_B.append(act_B); mem_val_B.append(value_B); mem_logp_B.append(log_prob_B)

            force_A_now, force_B_now = env.force_A, env.force_B
            can_pull_A_now, can_pull_B_now = state_A_dict['can_pull'], state_B_dict['can_pull']
            next_A, next_B, r_A, r_B, done = env.step(act_A, act_B)
            mem_rew_A.append(r_A); mem_rew_B.append(r_B)

            # >>> ONLY CHANGE IN THIS CELL: added A_gaze/B_gaze to the logged dict <
            log.append({'force_A': force_A_now, 'force_B': force_B_now,
                        'A_pull': act_A['pull'], 'B_pull': act_B['pull'],
                        'A_gaze': act_A['gaze'], 'B_gaze': act_B['gaze'],
                        'can_pull_A': can_pull_A_now, 'can_pull_B': can_pull_B_now})

            private_A = get_private_state(next_B, env.B_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_A['gaze']==1)
            private_B = get_private_state(next_A, env.A_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_B['gaze']==1)
            public_A = preprocess_state_with_effort(next_A, env.force_A, env.force_B)
            public_B = preprocess_state_with_effort(next_B, env.force_B, env.force_A)
            state_A_dict, state_B_dict = next_A, next_B

        with torch.no_grad():
            _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
            _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)
        returns_A = calculate_gae(mem_rew_A, mem_val_A, next_value_A)
        returns_B = calculate_gae(mem_rew_B, mem_val_B, next_value_B)
        values_A_old = torch.cat(mem_val_A).view(-1)
        values_B_old = torch.cat(mem_val_B).view(-1)
        adv_A = (returns_A - values_A_old); adv_A = (adv_A - adv_A.mean()) / (adv_A.std() + 1e-8)
        adv_B = (returns_B - values_B_old); adv_B = (adv_B - adv_B.mean()) / (adv_B.std() + 1e-8)

        curr_hidden_A, curr_hidden_B = start_hidden_A, start_hidden_B
        new_lp_A_list, new_lp_B_list = [], []
        entropy_A_list, entropy_B_list = [], []
        for t in range(N_STEPS):
            (l_A), _, curr_hidden_A = agent_A(mem_pub_A[t], mem_priv_A[t], curr_hidden_A, mem_mask_A[t])
            m,p,g = l_A; a = mem_act_A[t]
            new_lp_A_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_A = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()   # PULL-ONLY entropy
            entropy_A_list.append(ent_A)

            (l_B), _, curr_hidden_B = agent_B(mem_pub_B[t], mem_priv_B[t], curr_hidden_B, mem_mask_B[t])
            m,p,g = l_B; a = mem_act_B[t]
            new_lp_B_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_B = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()   # PULL-ONLY entropy
            entropy_B_list.append(ent_B)

        new_lp_A = torch.stack(new_lp_A_list).view(-1); old_lp_A = torch.cat(mem_logp_A).detach().view(-1)
        new_lp_B = torch.stack(new_lp_B_list).view(-1); old_lp_B = torch.cat(mem_logp_B).detach().view(-1)
        mean_ent_A = torch.stack(entropy_A_list).mean()
        mean_ent_B = torch.stack(entropy_B_list).mean()

        ratio_A = torch.exp(new_lp_A - old_lp_A)
        loss_A = -torch.min(ratio_A*adv_A, torch.clamp(ratio_A,1-EPS_CLIP,1+EPS_CLIP)*adv_A).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_A_old, returns_A) - entropy_coef_used * mean_ent_A
    
        ratio_B = torch.exp(new_lp_B - old_lp_B)
        loss_B = -torch.min(ratio_B*adv_B, torch.clamp(ratio_B,1-EPS_CLIP,1+EPS_CLIP)*adv_B).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_B_old, returns_B) - entropy_coef_used * mean_ent_B
        

        optimizer_A.zero_grad(); loss_A.backward(); torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 0.5); optimizer_A.step()
        optimizer_B.zero_grad(); loss_B.backward(); torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 0.5); optimizer_B.step()

        if update_step % (5000 // N_STEPS) == 0:
            steps_so_far = update_step * N_STEPS
            seed_tag = f" seed={seed}" if seed is not None else ""
            print(f"    [scale x{cost_scale_multiplier}{seed_tag}] step {steps_so_far}/{n_steps}, "
                  f"R_A: {sum(mem_rew_A):.2f}, R_B: {sum(mem_rew_B):.2f}")

    log_df = pd.DataFrame(log)
    eligible_A = log_df[log_df['can_pull_A']]
    eligible_B = log_df[log_df['can_pull_B']]

    mean_entropy_A_realized = mean_ent_A.item()
    mean_entropy_B_realized = mean_ent_B.item()
    
    corr_A = eligible_A['force_A'].corr(eligible_A['A_pull'])
    corr_B = eligible_B['force_B'].corr(eligible_B['B_pull'])
    pct_eligible_A = log_df['can_pull_A'].mean()
    pct_eligible_B = log_df['can_pull_B'].mean()
    attempt_rate_A = eligible_A['A_pull'].mean()
    attempt_rate_B = eligible_B['B_pull'].mean()
    return corr_A, corr_B, pct_eligible_A, pct_eligible_B, attempt_rate_A, attempt_rate_B, mean_entropy_A_realized, mean_entropy_B_realized, log_df


SWEEP_RESULTS_OUT = f"cost_scale_sweep_results_{CHECKPOINT_SET}.pkl"
N_REPLICATES = 3
SEED_LIST = list(range(N_REPLICATES))


def run_cost_scale_sweep(force_retrain=False, n_replicates=N_REPLICATES, seeds=None):
    seeds = seeds if seeds is not None else SEED_LIST[:n_replicates]

    if not force_retrain and os.path.exists(SWEEP_RESULTS_OUT):
        print(f"Found existing {SWEEP_RESULTS_OUT} -- loading instead of re-running.")
        print("Call run_cost_scale_sweep(force_retrain=True) to redo the sweep.\n")
        results_df = pd.read_pickle(SWEEP_RESULTS_OUT)
        print(results_df.to_string(index=False))
        return results_df

    if force_retrain and os.path.exists(SWEEP_RESULTS_OUT):
        print(f"force_retrain=True -- overwriting existing {SWEEP_RESULTS_OUT}\n")

    print(f"Running {len(SWEEP_SCALES)} multipliers x {len(seeds)} seeds each "
          f"({len(SWEEP_SCALES)*len(seeds)} total probes, checkpoint={CHECKPOINT_SET})\n")

    raw_results = []
    for mult in SWEEP_SCALES:
        for seed in seeds:
            print(f"\n=== multiplier x{mult}, seed {seed} ({SWEEP_STEPS} steps) ===")
            corr_A, corr_B, pct_elig_A, pct_elig_B, atr_A, atr_B, ent_A, ent_B, log_df = \
                run_short_probe(mult, seed=seed)
            max_cost = BASE_SCALE * mult * 900
            raw_results.append({
                'multiplier': mult, 'seed': seed, 'max_cost': max_cost,
                'corr_A': corr_A, 'corr_B': corr_B,
                'pct_eligible_A': pct_elig_A, 'pct_eligible_B': pct_elig_B,
                'attempt_rate_A': atr_A, 'attempt_rate_B': atr_B,
                'entropy_A': ent_A, 'entropy_B': ent_B,
            })
            log_df.to_pickle(f"cost_scale_sweep_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl")

    raw_df = pd.DataFrame(raw_results)
    raw_df.to_pickle(f"cost_scale_sweep_rawresults_{CHECKPOINT_SET}.pkl")

    agg = raw_df.groupby('multiplier').agg(
        max_cost=('max_cost', 'first'),
        corr_A_mean=('corr_A', 'mean'), corr_A_std=('corr_A', 'std'),
        corr_B_mean=('corr_B', 'mean'), corr_B_std=('corr_B', 'std'),
        attempt_rate_A_mean=('attempt_rate_A', 'mean'), attempt_rate_A_std=('attempt_rate_A', 'std'),
        attempt_rate_B_mean=('attempt_rate_B', 'mean'), attempt_rate_B_std=('attempt_rate_B', 'std'),
        entropy_A_mean=('entropy_A', 'mean'), entropy_A_std=('entropy_A', 'std'),
        entropy_B_mean=('entropy_B', 'mean'), entropy_B_std=('entropy_B', 'std'),
    ).reset_index()

    print("\n" + "="*100)
    print(f"AGGREGATED OVER {len(seeds)} SEEDS ({CHECKPOINT_SET}):")
    print(agg.to_string(index=False))
    print("="*100)

    agg.to_pickle(SWEEP_RESULTS_OUT)
    print(f"\nSaved {SWEEP_RESULTS_OUT} (aggregated) and "
          f"cost_scale_sweep_rawresults_{CHECKPOINT_SET}.pkl (per-seed)")
    return agg


# --- To run the sweep ---
# run_cost_scale_sweep()                                # skips if results already exist
# run_cost_scale_sweep(force_retrain=True)               # always re-runs, 3 seeds/multiplier
# run_cost_scale_sweep(force_retrain=True, n_replicates=5, seeds=[0,1,2,3,4])  # more replicates
results_df = run_cost_scale_sweep()

In [ ]:
"""
Plots raw conditional attempt rate (P(pull | eligible)) vs. force, one line
per cost-scale multiplier, using the raw per-seed logs already saved by the
sweep -- no retraining needed. Updated to match the new sweep:
  - CHECKPOINT_SET-aware filenames
  - wider SWEEP_SCALES range
  - multiple seeds per multiplier -> plots mean with a shaded std band
    across seeds, and prints mean +/- std for the overall attempt rate.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CHECKPOINT_SET = "generalist_withGAE_PPO"  # "generalist_withGAE_PPO" or "efficient_expert"
SWEEP_SCALES = [1.0, 3.0, 5.0, 8.0, 12.0, 16.0, 20.0, 25.0]   # wider range this time
SEED_LIST = [0, 1, 2]   # match N_REPLICATES / seeds used in the sweep

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

print(f"{'Multiplier':>10} | {'Attempt rate A (mean +/- std)':>30} | {'Attempt rate B (mean +/- std)':>30}")

for mult in SWEEP_SCALES:
    # --- Load all seeds for this multiplier ---
    per_seed_overall_A, per_seed_overall_B = [], []
    per_seed_binned_A, per_seed_binned_B = [], []

    for seed in SEED_LIST:
        fname = f"cost_scale_sweep_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            print(f"  (missing {fname}, skipping this seed)")
            continue

        elig_A = log_df[log_df['can_pull_A']].copy()
        elig_B = log_df[log_df['can_pull_B']].copy()

        per_seed_overall_A.append(elig_A['A_pull'].mean())
        per_seed_overall_B.append(elig_B['B_pull'].mean())

        elig_A['force_bin'] = (elig_A['force_A'] // 100) * 100
        elig_B['force_bin'] = (elig_B['force_B'] // 100) * 100
        per_seed_binned_A.append(elig_A.groupby('force_bin')['A_pull'].mean())
        per_seed_binned_B.append(elig_B.groupby('force_bin')['B_pull'].mean())

    if not per_seed_overall_A:
        print(f"{mult:>10} | {'NO DATA':>30} | {'NO DATA':>30}")
        continue

    mean_A, std_A = np.mean(per_seed_overall_A), np.std(per_seed_overall_A)
    mean_B, std_B = np.mean(per_seed_overall_B), np.std(per_seed_overall_B)
    print(f"{mult:>10} | {mean_A:>18.4f} +/- {std_A:<9.4f} | {mean_B:>18.4f} +/- {std_B:<9.4f}")

    # --- Average the binned curves across seeds (align on force_bin index) ---
    binned_A_df = pd.concat(per_seed_binned_A, axis=1)
    binned_B_df = pd.concat(per_seed_binned_B, axis=1)
    mean_curve_A, std_curve_A = binned_A_df.mean(axis=1), binned_A_df.std(axis=1)
    mean_curve_B, std_curve_B = binned_B_df.mean(axis=1), binned_B_df.std(axis=1)

    line_A, = axes[0].plot(mean_curve_A.index, mean_curve_A.values, 'o-', label=f'x{mult}')
    axes[0].fill_between(mean_curve_A.index, mean_curve_A.values - std_curve_A.values,
                          mean_curve_A.values + std_curve_A.values, alpha=0.15, color=line_A.get_color())

    line_B, = axes[1].plot(mean_curve_B.index, mean_curve_B.values, 'o-', label=f'x{mult}')
    axes[1].fill_between(mean_curve_B.index, mean_curve_B.values - std_curve_B.values,
                          mean_curve_B.values + std_curve_B.values, alpha=0.15, color=line_B.get_color())

axes[0].set_title(f"Agent A: P(pull | eligible) vs own force ({CHECKPOINT_SET}, {len(SEED_LIST)} seeds)")
axes[0].set_xlabel("Force (g)")
axes[0].set_ylabel("P(pull | eligible)")
axes[0].set_ylim(0, 1.05)
axes[0].legend(title="Cost scale")

axes[1].set_title(f"Agent B: P(pull | eligible) vs own force ({CHECKPOINT_SET}, {len(SEED_LIST)} seeds)")
axes[1].set_xlabel("Force (g)")
axes[1].set_ylabel("P(pull | eligible)")
axes[1].set_ylim(0, 1.05)
axes[1].legend(title="Cost scale")

plt.tight_layout()
plt.savefig(f"attempt_rate_ceiling_check_{CHECKPOINT_SET}.pdf", dpi=150)
print(f"\nSaved attempt_rate_ceiling_check_{CHECKPOINT_SET}.pdf")

print("\nShaded bands = +/- 1 std across seeds. If bands are wide relative to the")
print("gap between multiplier lines, differences between cost levels aren't")
print("distinguishable from run-to-run noise yet. If all lines/bands sit near 1.0")
print("regardless of force or multiplier, that confirms the ceiling-effect")
print("explanation: agents attempt on essentially every eligible step no matter")
print("the cost, because a missed attempt guarantees zero reward while an attempt")
print("only risks a modest cost against a much larger potential payoff.")

In [ ]:
"""
Inter-pull-interval (IPI) vs. own force, by cost-scale multiplier.
Uses the raw per-step sweep logs already saved -- no retraining needed.

Rationale: attempt-rate-per-eligible-step only captures whether an agent
ever pulls, not WHEN. A cost-sensitive strategy could show up as "pull less
promptly" (longer gaps between pulls) rather than "pull less often" -- this
would be invisible to the attempt-rate metric but visible here.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CHECKPOINT_SET = "generalist_withGAE_PPO"  # "generalist_withGAE_PPO" or "efficient_expert"
SWEEP_SCALES = [1.0, 3.0, 5.0, 8.0, 12.0, 16.0, 20.0, 25.0]   # match the full sweep, not the trimmed plot
# SWEEP_SCALES = [1.0, 3.0, 5.0, 8.0, 12.0, 25.0]
SEED_LIST = [0, 1, 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

print(f"{'Multiplier':>10} | {'Mean IPI A (steps, mean +/- std)':>34} | {'Mean IPI B (steps, mean +/- std)':>34}")

for mult in SWEEP_SCALES:
    per_seed_overall_ipi_A, per_seed_overall_ipi_B = [], []
    per_seed_binned_ipi_A, per_seed_binned_ipi_B = [], []

    for seed in SEED_LIST:
        fname = f"cost_scale_sweep_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            print(f"  (missing {fname}, skipping this seed)")
            continue

        for agent, pull_col, can_pull_col, force_col, overall_list, binned_list in [
            ('A', 'A_pull', 'can_pull_A', 'force_A', per_seed_overall_ipi_A, per_seed_binned_ipi_A),
            ('B', 'B_pull', 'can_pull_B', 'force_B', per_seed_overall_ipi_B, per_seed_binned_ipi_B),
        ]:
            # Actual executed pulls: action==1 AND eligible (mask allowed it through)
            pull_idx = log_df.index[(log_df[pull_col] == 1) & (log_df[can_pull_col])].to_numpy()
            if len(pull_idx) < 2:
                continue

            ipi = np.diff(pull_idx)  # steps between consecutive pulls
            overall_list.append(ipi.mean())

            # Bin each IPI by the force active at the START of that interval
            force_at_start = log_df.loc[pull_idx[:-1], force_col].to_numpy()
            ipi_df = pd.DataFrame({'force_bin': (force_at_start // 100) * 100, 'ipi': ipi})
            binned_list.append(ipi_df.groupby('force_bin')['ipi'].mean())

    if not per_seed_overall_ipi_A:
        print(f"{mult:>10} | {'NO DATA':>34} | {'NO DATA':>34}")
        continue

    mean_A, std_A = np.mean(per_seed_overall_ipi_A), np.std(per_seed_overall_ipi_A)
    mean_B, std_B = (np.mean(per_seed_overall_ipi_B), np.std(per_seed_overall_ipi_B)) \
        if per_seed_overall_ipi_B else (np.nan, np.nan)
    print(f"{mult:>10} | {mean_A:>22.2f} +/- {std_A:<7.2f} | {mean_B:>22.2f} +/- {std_B:<7.2f}")

    if per_seed_binned_ipi_A:
        binned_A_df = pd.concat(per_seed_binned_ipi_A, axis=1)
        mean_curve_A, std_curve_A = binned_A_df.mean(axis=1), binned_A_df.std(axis=1)
        line_A, = axes[0].plot(mean_curve_A.index, mean_curve_A.values, 'o-', label=f'x{mult}')
        axes[0].fill_between(mean_curve_A.index, mean_curve_A.values - std_curve_A.values,
                              mean_curve_A.values + std_curve_A.values, alpha=0.15, color=line_A.get_color())

    if per_seed_binned_ipi_B:
        binned_B_df = pd.concat(per_seed_binned_ipi_B, axis=1)
        mean_curve_B, std_curve_B = binned_B_df.mean(axis=1), binned_B_df.std(axis=1)
        line_B, = axes[1].plot(mean_curve_B.index, mean_curve_B.values, 'o-', label=f'x{mult}')
        axes[1].fill_between(mean_curve_B.index, mean_curve_B.values - std_curve_B.values,
                              mean_curve_B.values + std_curve_B.values, alpha=0.15, color=line_B.get_color())

axes[0].set_title(f"Agent A: mean inter-pull interval vs own force ({CHECKPOINT_SET}, {len(SEED_LIST)} seeds)")
axes[0].set_xlabel("Force (g)")
axes[0].set_ylabel("Mean IPI (steps)")
axes[0].legend(title="Cost scale")

axes[1].set_title(f"Agent B: mean inter-pull interval vs own force ({CHECKPOINT_SET}, {len(SEED_LIST)} seeds)")
axes[1].set_xlabel("Force (g)")
axes[1].set_ylabel("Mean IPI (steps)")
axes[1].legend(title="Cost scale")

plt.tight_layout()
plt.savefig(f"ipi_vs_force_{CHECKPOINT_SET}.pdf", dpi=150)
print(f"\nSaved ipi_vs_force_{CHECKPOINT_SET}.pdf")

print("\nIf IPI increases with force (agent takes longer between pulls as its own")
print("cost rises), that's cost-sensitivity showing up in TIMING rather than in")
print("raw attempt rate -- a real effect the eligible-attempt-rate metric would miss.")
print("Note: B's ~1.0 attempt rate means B pulls almost every eligible step, so its")
print("IPI is mechanically close to the cooldown/drink-timer cycle length regardless")
print("of cost -- if B's IPI is also flat, that's consistent with B being genuinely")
print("insensitive to cost, not just insensitive on the wrong metric.")

In [ ]:
"""
Gaze rate vs. own force, by cost-scale multiplier -- for Agent A (the
non-saturated/adaptive agent, per your role assumption).

Tests a different hypothesis than attempt rate: instead of "does cost make
A pull less," this checks "does cost make A gather more evidence (gaze more)
before committing" -- closer to your original gaze-dependent-accumulation
finding from the neuron paper. Unlike pull, gaze isn't masked by eligibility
(the agent can gaze whether or not it's able to pull), so this is NOT
conditioned on can_pull -- it's the raw P(gaze=1) at every step.

Requires raw logs generated AFTER the A_gaze/B_gaze logging fix -- rerun
run_cost_scale_sweep(force_retrain=True) first if your saved logs predate it.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CHECKPOINT_SET = "generalist_withGAE_PPO"   # match whatever you actually swept
SWEEP_SCALES = [1.0, 3.0, 5.0, 8.0, 12.0, 16.0, 20.0, 25.0]
SEED_LIST = [0, 1, 2]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

print(f"{'Multiplier':>10} | {'Gaze rate A (mean +/- std)':>28} | {'Gaze rate B (mean +/- std)':>28}")

for mult in SWEEP_SCALES:
    per_seed_overall_A, per_seed_overall_B = [], []
    per_seed_binned_A, per_seed_binned_B = [], []

    for seed in SEED_LIST:
        fname = f"cost_scale_sweep_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            print(f"  (missing {fname}, skipping this seed)")
            continue

        if 'A_gaze' not in log_df.columns:
            print(f"  ({fname} predates the gaze-logging fix -- rerun with force_retrain=True)")
            continue

        per_seed_overall_A.append(log_df['A_gaze'].mean())
        per_seed_overall_B.append(log_df['B_gaze'].mean())

        log_df = log_df.copy()
        log_df['force_bin_A'] = (log_df['force_A'] // 100) * 100
        log_df['force_bin_B'] = (log_df['force_B'] // 100) * 100
        per_seed_binned_A.append(log_df.groupby('force_bin_A')['A_gaze'].mean())
        per_seed_binned_B.append(log_df.groupby('force_bin_B')['B_gaze'].mean())

    if not per_seed_overall_A:
        print(f"{mult:>10} | {'NO DATA':>28} | {'NO DATA':>28}")
        continue

    mean_A, std_A = np.mean(per_seed_overall_A), np.std(per_seed_overall_A)
    mean_B, std_B = np.mean(per_seed_overall_B), np.std(per_seed_overall_B)
    print(f"{mult:>10} | {mean_A:>18.4f} +/- {std_A:<7.4f} | {mean_B:>18.4f} +/- {std_B:<7.4f}")

    binned_A_df = pd.concat(per_seed_binned_A, axis=1)
    mean_curve_A, std_curve_A = binned_A_df.mean(axis=1), binned_A_df.std(axis=1)
    line_A, = axes[0].plot(mean_curve_A.index, mean_curve_A.values, 'o-', label=f'x{mult}')
    axes[0].fill_between(mean_curve_A.index, mean_curve_A.values - std_curve_A.values,
                          mean_curve_A.values + std_curve_A.values, alpha=0.15, color=line_A.get_color())

    binned_B_df = pd.concat(per_seed_binned_B, axis=1)
    mean_curve_B, std_curve_B = binned_B_df.mean(axis=1), binned_B_df.std(axis=1)
    line_B, = axes[1].plot(mean_curve_B.index, mean_curve_B.values, 'o-', label=f'x{mult}')
    axes[1].fill_between(mean_curve_B.index, mean_curve_B.values - std_curve_B.values,
                          mean_curve_B.values + std_curve_B.values, alpha=0.15, color=line_B.get_color())

axes[0].set_title(f"Agent A (adaptive): P(gaze) vs own force ({CHECKPOINT_SET})")
axes[0].set_xlabel("Force (g)")
axes[0].set_ylabel("P(gaze=1)")
axes[0].set_ylim(0, 1.05)
axes[0].legend(title="Cost scale", fontsize=8)

axes[1].set_title(f"Agent B (anchor): P(gaze) vs own force ({CHECKPOINT_SET})")
axes[1].set_xlabel("Force (g)")
axes[1].set_ylabel("P(gaze=1)")
axes[1].set_ylim(0, 1.05)
axes[1].legend(title="Cost scale", fontsize=8)

plt.tight_layout()
plt.savefig(f"gaze_rate_vs_force_{CHECKPOINT_SET}.png", dpi=150)
print(f"\nSaved gaze_rate_vs_force_{CHECKPOINT_SET}.png")
print("\nIf Agent A's gaze rate climbs with its own force, that's evidence the adaptive")
print("agent shifts to an 'accumulate more evidence before acting' strategy under cost,")
print("even if raw attempt rate itself doesn't move much -- a more direct match to your")
print("gaze-dependent-accumulation hypothesis than attempt rate alone.")

### run the training with different sensitivity to the effort level to figure out the best choice
### only change agent A's cost, and shorten the block length

In [ ]:
SWEEP_STEPS = 400000   # was 100000

def run_short_probe_fixed_role(cost_scale_multiplier, n_steps=SWEEP_STEPS, seed=None,
                                 entropy_coef=None, block_len=1200, lock_fixed_agent='B'):
    """
    Same as run_short_probe, but:
    - lock_fixed_agent='B': Agent A is ALWAYS the varying agent, Agent B ALWAYS fixed
      (no session-flip dilution -- every block tests A's own-cost sensitivity)
    - block_len=1200 (vs 3000): ~2.5x more distinct force samples per 100k-step run
    """
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    entropy_coef_used = entropy_coef if entropy_coef is not None else ENTROPY_COEF
    global FORCE_TO_COST_SCALE
    FORCE_TO_COST_SCALE = BASE_SCALE * cost_scale_multiplier

    env = CurriculumEnv(block_len=block_len, lock_fixed_agent=lock_fixed_agent)   # <-- only line that changed
    env.set_training_phase("COOP_FULL_RANDOM", sync_window=TARGET_WINDOW)
    env.gaze_cost = GAZE_COST

    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_A = load_and_expand_agent(agent_A, f"{OLD_CHECKPOINT_DIR}/agent_A_{CHECKPOINT_SET}.pth")
    agent_B = load_and_expand_agent(agent_B, f"{OLD_CHECKPOINT_DIR}/agent_B_{CHECKPOINT_SET}.pth")

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state_with_effort(state_A_dict, env.force_A, env.force_B)
    public_B = preprocess_state_with_effort(state_B_dict, env.force_B, env.force_A)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)

    log = []

    for update_step in range(n_steps // N_STEPS):
        mem_pub_A, mem_priv_A, mem_mask_A = [], [], []
        mem_act_A, mem_val_A, mem_rew_A, mem_logp_A = [], [], [], []
        mem_pub_B, mem_priv_B, mem_mask_B = [], [], []
        mem_act_B, mem_val_B, mem_rew_B, mem_logp_B = [], [], [], []

        start_hidden_A, start_hidden_B = hidden_A.detach(), hidden_B.detach()
        hidden_A, hidden_B = start_hidden_A, start_hidden_B

        for _ in range(N_STEPS):
            mask_A = torch.tensor([[0., 0. if state_A_dict['can_pull'] else -1e9]], device=device).float()
            mask_B = torch.tensor([[0., 0. if state_B_dict['can_pull'] else -1e9]], device=device).float()

            mem_pub_A.append(public_A); mem_priv_A.append(private_A); mem_mask_A.append(mask_A)
            mem_pub_B.append(public_B); mem_priv_B.append(private_B); mem_mask_B.append(mask_B)

            with torch.no_grad():
                logits_A, value_A, hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
                logits_B, value_B, hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)

            act_A, log_prob_A = sample_actions(logits_A)
            act_B, log_prob_B = sample_actions(logits_B)
            mem_act_A.append(act_A); mem_val_A.append(value_A); mem_logp_A.append(log_prob_A)
            mem_act_B.append(act_B); mem_val_B.append(value_B); mem_logp_B.append(log_prob_B)

            force_A_now, force_B_now = env.force_A, env.force_B
            can_pull_A_now, can_pull_B_now = state_A_dict['can_pull'], state_B_dict['can_pull']
            next_A, next_B, r_A, r_B, done = env.step(act_A, act_B)
            mem_rew_A.append(r_A); mem_rew_B.append(r_B)

            log.append({'force_A': force_A_now, 'force_B': force_B_now,
                        'A_pull': act_A['pull'], 'B_pull': act_B['pull'],
                        'A_gaze': act_A['gaze'], 'B_gaze': act_B['gaze'],
                        'can_pull_A': can_pull_A_now, 'can_pull_B': can_pull_B_now})

            private_A = get_private_state(next_B, env.B_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_A['gaze']==1)
            private_B = get_private_state(next_A, env.A_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_B['gaze']==1)
            public_A = preprocess_state_with_effort(next_A, env.force_A, env.force_B)
            public_B = preprocess_state_with_effort(next_B, env.force_B, env.force_A)
            state_A_dict, state_B_dict = next_A, next_B

        with torch.no_grad():
            _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
            _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)
        returns_A = calculate_gae(mem_rew_A, mem_val_A, next_value_A)
        returns_B = calculate_gae(mem_rew_B, mem_val_B, next_value_B)
        values_A_old = torch.cat(mem_val_A).view(-1)
        values_B_old = torch.cat(mem_val_B).view(-1)
        adv_A = (returns_A - values_A_old); adv_A = (adv_A - adv_A.mean()) / (adv_A.std() + 1e-8)
        adv_B = (returns_B - values_B_old); adv_B = (adv_B - adv_B.mean()) / (adv_B.std() + 1e-8)

        curr_hidden_A, curr_hidden_B = start_hidden_A, start_hidden_B
        new_lp_A_list, new_lp_B_list = [], []
        entropy_A_list, entropy_B_list = [], []
        for t in range(N_STEPS):
            (l_A), _, curr_hidden_A = agent_A(mem_pub_A[t], mem_priv_A[t], curr_hidden_A, mem_mask_A[t])
            m,p,g = l_A; a = mem_act_A[t]
            new_lp_A_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_A = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()
            entropy_A_list.append(ent_A)

            (l_B), _, curr_hidden_B = agent_B(mem_pub_B[t], mem_priv_B[t], curr_hidden_B, mem_mask_B[t])
            m,p,g = l_B; a = mem_act_B[t]
            new_lp_B_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_B = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()
            entropy_B_list.append(ent_B)

        new_lp_A = torch.stack(new_lp_A_list).view(-1); old_lp_A = torch.cat(mem_logp_A).detach().view(-1)
        new_lp_B = torch.stack(new_lp_B_list).view(-1); old_lp_B = torch.cat(mem_logp_B).detach().view(-1)
        mean_ent_A = torch.stack(entropy_A_list).mean()
        mean_ent_B = torch.stack(entropy_B_list).mean()

        ratio_A = torch.exp(new_lp_A - old_lp_A)
        loss_A = -torch.min(ratio_A*adv_A, torch.clamp(ratio_A,1-EPS_CLIP,1+EPS_CLIP)*adv_A).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_A_old, returns_A) - entropy_coef_used * mean_ent_A
        ratio_B = torch.exp(new_lp_B - old_lp_B)
        loss_B = -torch.min(ratio_B*adv_B, torch.clamp(ratio_B,1-EPS_CLIP,1+EPS_CLIP)*adv_B).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_B_old, returns_B) - entropy_coef_used * mean_ent_B

        optimizer_A.zero_grad(); loss_A.backward(); torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 0.5); optimizer_A.step()
        optimizer_B.zero_grad(); loss_B.backward(); torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 0.5); optimizer_B.step()

        if update_step % (5000 // N_STEPS) == 0:
            steps_so_far = update_step * N_STEPS
            print(f"    [scale x{cost_scale_multiplier} seed={seed} FIXED-ROLE] step {steps_so_far}/{n_steps}, "
                  f"R_A: {sum(mem_rew_A):.2f}, R_B: {sum(mem_rew_B):.2f}")

    log_df = pd.DataFrame(log)
    eligible_A = log_df[log_df['can_pull_A']]
    eligible_B = log_df[log_df['can_pull_B']]
    mean_entropy_A_realized = mean_ent_A.item()
    mean_entropy_B_realized = mean_ent_B.item()
    corr_A = eligible_A['force_A'].corr(eligible_A['A_pull'])
    corr_B = eligible_B['force_B'].corr(eligible_B['B_pull'])
    attempt_rate_A = eligible_A['A_pull'].mean()
    attempt_rate_B = eligible_B['B_pull'].mean()
    n_blocks = n_steps // block_len
    return corr_A, corr_B, attempt_rate_A, attempt_rate_B, mean_entropy_A_realized, mean_entropy_B_realized, n_blocks, log_df


FIXEDROLE_RESULTS_OUT = f"cost_scale_sweep_fixedrole_results_{CHECKPOINT_SET}.pkl"

def run_cost_scale_sweep_fixed_role(force_retrain=False, n_replicates=3, seeds=None,
                                      block_len=1200, lock_fixed_agent='B'):
    seeds = seeds if seeds is not None else list(range(n_replicates))

    if not force_retrain and os.path.exists(FIXEDROLE_RESULTS_OUT):
        print(f"Found existing {FIXEDROLE_RESULTS_OUT} -- loading instead of re-running.")
        results_df = pd.read_pickle(FIXEDROLE_RESULTS_OUT)
        print(results_df.to_string(index=False))
        return results_df

    print(f"Running FIXED-ROLE sweep: A always varying, B always fixed, block_len={block_len} "
          f"({SWEEP_STEPS//block_len} blocks/run vs "
          f"{SWEEP_STEPS//3000} in the original design)\n")

    raw_results = []
    for mult in SWEEP_SCALES:
        for seed in seeds:
            print(f"\n=== multiplier x{mult}, seed {seed}, FIXED-ROLE ({SWEEP_STEPS} steps) ===")
            corr_A, corr_B, atr_A, atr_B, ent_A, ent_B, n_blocks, log_df = \
                run_short_probe_fixed_role(mult, seed=seed, block_len=block_len, lock_fixed_agent=lock_fixed_agent)
            max_cost = BASE_SCALE * mult * 900
            raw_results.append({
                'multiplier': mult, 'seed': seed, 'max_cost': max_cost,
                'corr_A': corr_A, 'corr_B': corr_B,
                'attempt_rate_A': atr_A, 'attempt_rate_B': atr_B,
                'entropy_A': ent_A, 'entropy_B': ent_B, 'n_blocks': n_blocks,
            })
            log_df.to_pickle(f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl")

    raw_df = pd.DataFrame(raw_results)
    agg = raw_df.groupby('multiplier').agg(
        max_cost=('max_cost', 'first'),
        corr_A_mean=('corr_A', 'mean'), corr_A_std=('corr_A', 'std'),
        attempt_rate_A_mean=('attempt_rate_A', 'mean'), attempt_rate_A_std=('attempt_rate_A', 'std'),
        entropy_A_mean=('entropy_A', 'mean'), entropy_A_std=('entropy_A', 'std'),
    ).reset_index()

    print("\n" + "="*90)
    print(f"FIXED-ROLE SWEEP RESULTS (A always varying, B always fixed, block_len={block_len}):")
    print(agg.to_string(index=False))
    print("="*90)

    agg.to_pickle(FIXEDROLE_RESULTS_OUT)
    raw_df.to_pickle(f"cost_scale_sweep_fixedrole_rawresults_{CHECKPOINT_SET}.pkl")
    return agg

# run_cost_scale_sweep_fixed_role(force_retrain=True)               # always re-runs, 3 seeds/multiplier
fixedrole_results_df = run_cost_scale_sweep_fixed_role()

In [ ]:
#################################################################
# FIXED-ROLE SWEEP RESULTS: attempt rate, IPI, and gaze rate vs force (Agent A)
#################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

FIXEDROLE_SWEEP_SCALES = SWEEP_SCALES   # reuse whatever was defined earlier in the notebook
FIXEDROLE_SEED_LIST = list(range(3))     # matches n_replicates=3 default used in the sweep

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

print(f"{'Multiplier':>10} | {'Attempt rate A':>22} | {'Mean IPI A (steps)':>22} | {'Gaze rate A':>18}")

for mult in FIXEDROLE_SWEEP_SCALES:
    per_seed_overall_attempt, per_seed_binned_attempt = [], []
    per_seed_overall_ipi, per_seed_binned_ipi = [], []
    per_seed_overall_gaze, per_seed_binned_gaze = [], []

    for seed in FIXEDROLE_SEED_LIST:
        fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            print(f"  (missing {fname}, skipping this seed)")
            continue

        log_df = log_df.copy()
        log_df['force_bin'] = (log_df['force_A'] // 100) * 100

        # --- 1. Attempt rate (conditioned on eligibility) ---
        eligible = log_df[log_df['can_pull_A']]
        if len(eligible) > 0:
            per_seed_overall_attempt.append(eligible['A_pull'].mean())
            per_seed_binned_attempt.append(eligible.groupby('force_bin')['A_pull'].mean())

        # --- 2. Inter-pull interval (IPI) ---
        pull_idx = log_df.index[(log_df['A_pull'] == 1) & (log_df['can_pull_A'])].to_numpy()
        if len(pull_idx) >= 2:
            ipi = np.diff(pull_idx)
            per_seed_overall_ipi.append(ipi.mean())
            force_at_start = log_df.loc[pull_idx[:-1], 'force_A'].to_numpy()
            ipi_df = pd.DataFrame({'force_bin': (force_at_start // 100) * 100, 'ipi': ipi})
            per_seed_binned_ipi.append(ipi_df.groupby('force_bin')['ipi'].mean())

        # --- 3. Gaze rate (unconditioned -- gaze isn't masked by eligibility) ---
        if 'A_gaze' in log_df.columns:
            per_seed_overall_gaze.append(log_df['A_gaze'].mean())
            per_seed_binned_gaze.append(log_df.groupby('force_bin')['A_gaze'].mean())

    # --- Print summary row ---
    atr_str = f"{np.mean(per_seed_overall_attempt):.4f} +/- {np.std(per_seed_overall_attempt):.4f}" if per_seed_overall_attempt else "NO DATA"
    ipi_str = f"{np.mean(per_seed_overall_ipi):.2f} +/- {np.std(per_seed_overall_ipi):.2f}" if per_seed_overall_ipi else "NO DATA"
    gaze_str = f"{np.mean(per_seed_overall_gaze):.4f} +/- {np.std(per_seed_overall_gaze):.4f}" if per_seed_overall_gaze else "NO DATA"
    print(f"{mult:>10} | {atr_str:>22} | {ipi_str:>22} | {gaze_str:>18}")

    # --- Plot each panel: mean curve + shaded std band across seeds ---
    def plot_panel(ax, per_seed_binned, label):
        if not per_seed_binned:
            return
        binned_df = pd.concat(per_seed_binned, axis=1)
        mean_curve, std_curve = binned_df.mean(axis=1), binned_df.std(axis=1)
        line, = ax.plot(mean_curve.index, mean_curve.values, 'o-', label=label)
        ax.fill_between(mean_curve.index, mean_curve.values - std_curve.values,
                         mean_curve.values + std_curve.values, alpha=0.15, color=line.get_color())

    plot_panel(axes[0], per_seed_binned_attempt, f'x{mult}')
    plot_panel(axes[1], per_seed_binned_ipi, f'x{mult}')
    plot_panel(axes[2], per_seed_binned_gaze, f'x{mult}')

axes[0].set_title(f"Agent A: attempt rate vs own force\n({CHECKPOINT_SET}, fixed-role, block_len={1200})")
axes[0].set_xlabel("Force (g)")
axes[0].set_ylabel("P(pull | eligible)")
axes[0].set_ylim(0, 1.05)
axes[0].legend(title="Cost scale", fontsize=7)

axes[1].set_title(f"Agent A: mean inter-pull interval vs own force\n({CHECKPOINT_SET}, fixed-role)")
axes[1].set_xlabel("Force (g)")
axes[1].set_ylabel("Mean IPI (steps)")
axes[1].legend(title="Cost scale", fontsize=7)

axes[2].set_title(f"Agent A: gaze rate vs own force\n({CHECKPOINT_SET}, fixed-role)")
axes[2].set_xlabel("Force (g)")
axes[2].set_ylabel("P(gaze=1)")
axes[2].set_ylim(0, 1.05)
axes[2].legend(title="Cost scale", fontsize=7)

plt.tight_layout()
plt.savefig(f"fixedrole_agentA_summary_{CHECKPOINT_SET}.png", dpi=150)
print(f"\nSaved fixedrole_agentA_summary_{CHECKPOINT_SET}.png")

In [ ]:
#################################################################
# PRECISE CHECK: regression of attempt rate on force, per multiplier
#################################################################
import pandas as pd
import numpy as np
from scipy import stats

print(f"{'Multiplier':>10} | {'Slope (per 100g)':>18} | {'p-value':>10} | {'Direction':>12} | {'n (eligible steps)':>18}")

for mult in FIXEDROLE_SWEEP_SCALES:
    all_force, all_pull = [], []

    for seed in FIXEDROLE_SEED_LIST:
        fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            continue
        eligible = log_df[log_df['can_pull_A']]
        all_force.extend(eligible['force_A'].tolist())
        all_pull.extend(eligible['A_pull'].tolist())

    if len(all_force) < 10:
        print(f"{mult:>10} | {'NOT ENOUGH DATA':>18}")
        continue

    all_force = np.array(all_force)
    all_pull = np.array(all_pull)

    # Logistic-style check via simple linear regression on the binary outcome
    # (a linear probability model -- fine for direction/significance here)
    slope, intercept, r_value, p_value, std_err = stats.linregress(all_force, all_pull)
    slope_per_100g = slope * 100

    direction = "predicted" if slope < 0 else "REVERSED"
    sig = "*" if p_value < 0.05 else " "
    print(f"{mult:>10} | {slope_per_100g:>17.5f}{sig} | {p_value:>10.4f} | {direction:>12} | {len(all_force):>18}")

print("\n'predicted' = higher force -> lower attempt rate (matches your hypothesis)")
print("'REVERSED' = higher force -> HIGHER attempt rate (opposite of hypothesis)")
print("* = statistically significant at p<0.05")
print("\nThis pools ALL raw eligible steps across seeds for each multiplier -- much more")
print("statistical power than reading 8 noisy force-bin means off a plot, and gives an")
print("unambiguous sign + significance rather than an eyeballed slope.")


In [ ]:
#################################################################
# DISENTANGLE: does force predict attempt rate independent of
# training-step position (time), or is the "reversed" effect
# actually a slow policy drift that only looks force-related?
#################################################################
import pandas as pd
import numpy as np

def multiple_regression_with_pvalues(X, y):
    """
    Simple OLS with intercept, returns coefficients, std errors, t-stats, p-values.
    X: 2D array (n_samples, n_features), y: 1D array.
    """
    n, k = X.shape
    X_design = np.column_stack([np.ones(n), X])  # add intercept column
    XtX_inv = np.linalg.inv(X_design.T @ X_design)
    beta = XtX_inv @ X_design.T @ y
    residuals = y - X_design @ beta
    sigma2 = (residuals @ residuals) / (n - k - 1)
    se = np.sqrt(np.diag(sigma2 * XtX_inv))
    t_stats = beta / se
    from scipy import stats as scipy_stats
    p_values = 2 * (1 - scipy_stats.t.cdf(np.abs(t_stats), df=n - k - 1))
    return beta, se, t_stats, p_values


print(f"{'Multiplier':>10} | {'Force coef (ctrl step)':>22} | {'Force p-value':>13} | {'Step coef':>12} | {'Step p-value':>13}")

for mult in FIXEDROLE_SWEEP_SCALES:
    all_force, all_pull, all_step = [], [], []

    for seed in FIXEDROLE_SEED_LIST:
        fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            continue
        eligible = log_df[log_df['can_pull_A']].copy()
        # Normalize step position within THIS seed's run to [0,1], so different
        # seeds' step counts combine sensibly rather than being on different scales
        eligible['step_frac'] = eligible.index / len(log_df)
        all_force.extend(eligible['force_A'].tolist())
        all_pull.extend(eligible['A_pull'].tolist())
        all_step.extend(eligible['step_frac'].tolist())

    if len(all_force) < 20:
        print(f"{mult:>10} | {'NOT ENOUGH DATA':>22}")
        continue

    X = np.column_stack([np.array(all_force), np.array(all_step)])
    y = np.array(all_pull)

    beta, se, t_stats, p_values = multiple_regression_with_pvalues(X, y)
    # beta[0]=intercept, beta[1]=force coefficient, beta[2]=step coefficient
    force_coef, force_p = beta[1] * 100, p_values[1]   # scaled per 100g
    step_coef, step_p = beta[2], p_values[2]

    sig_force = "*" if force_p < 0.05 else " "
    sig_step = "*" if step_p < 0.05 else " "
    print(f"{mult:>10} | {force_coef:>21.5f}{sig_force} | {force_p:>13.4f} | {step_coef:>11.4f}{sig_step} | {step_p:>13.4f}")

print("\nForce coefficient = effect of force on attempt rate, CONTROLLING for step position.")
print("Step coefficient = effect of time/training-progress on attempt rate, CONTROLLING for force.")
print("\nIf force stays POSITIVE and significant even after controlling for step: real,")
print("force-specific invigoration effect (not just a training-time drift).")
print("If force's effect shrinks/vanishes once step is included, or step itself is the")
print("significant predictor: the earlier 'reversed' result was likely confounded with")
print("policy drift over training time, not a genuine force-tracking effect.")

In [ ]:
#################################################################
# PLATEAU CHECK: attempt rate vs. training step (force ignored),
# to see if the "step effect" is still rising or has settled by
# the end of the 100k-step probe
#################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

STEP_BIN_SIZE = 5000   # bin width in raw training steps

fig, ax = plt.subplots(figsize=(12, 7))

print(f"{'Multiplier':>10} | {'Attempt rate: first 5k steps':>28} | {'Attempt rate: last 5k steps':>28} | {'Still rising?':>14}")

for mult in FIXEDROLE_SWEEP_SCALES:
    per_seed_binned = []
    first_vals, last_vals = [], []

    for seed in FIXEDROLE_SEED_LIST:
        fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{mult}_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            continue

        eligible = log_df[log_df['can_pull_A']].copy()
        if len(eligible) == 0:
            continue

        # Use the raw row index as the step number (log was appended in order)
        eligible['step_bin'] = (eligible.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
        binned = eligible.groupby('step_bin')['A_pull'].mean()
        per_seed_binned.append(binned)

        # First and last 5k-step windows, for the summary table
        n_total = len(log_df)
        first_window = eligible[eligible.index < STEP_BIN_SIZE]
        last_window = eligible[eligible.index >= n_total - STEP_BIN_SIZE]
        if len(first_window) > 0: first_vals.append(first_window['A_pull'].mean())
        if len(last_window) > 0: last_vals.append(last_window['A_pull'].mean())

    if not per_seed_binned:
        print(f"{mult:>10} | {'NO DATA':>28} | {'NO DATA':>28} |")
        continue

    binned_df = pd.concat(per_seed_binned, axis=1)
    mean_curve, std_curve = binned_df.mean(axis=1), binned_df.std(axis=1)

    line, = ax.plot(mean_curve.index, mean_curve.values, 'o-', label=f'x{mult}')
    ax.fill_between(mean_curve.index, mean_curve.values - std_curve.values,
                     mean_curve.values + std_curve.values, alpha=0.12, color=line.get_color())

    first_mean = np.mean(first_vals) if first_vals else float('nan')
    last_mean = np.mean(last_vals) if last_vals else float('nan')
    still_rising = "YES" if (last_mean - first_mean) > 0.02 else ("flat/settled" if abs(last_mean-first_mean) <= 0.02 else "declining")
    print(f"{mult:>10} | {first_mean:>28.4f} | {last_mean:>28.4f} | {still_rising:>14}")

ax.set_title(f"Agent A: attempt rate vs. training step (FORCE IGNORED)\n({CHECKPOINT_SET}, fixed-role)")
ax.set_xlabel("Training step (within the 100k-step probe)")
ax.set_ylabel("P(pull | eligible)")
ax.legend(title="Cost scale", fontsize=8)

plt.tight_layout()
plt.savefig(f"plateau_check_{CHECKPOINT_SET}.png", dpi=150)
print(f"\nSaved plateau_check_{CHECKPOINT_SET}.png")
print("\nIf the curves are still trending upward at step 100,000 (not flattening),")
print("that confirms the probe hasn't run long enough for the new force-input weights")
print("to finish settling -- the fix would be a longer probe, not a different mechanism.")
print("If curves already plateau well before the end, the drift is a converged bias,")
print("not just an unfinished warm-up -- a different and more concerning explanation.")

In [ ]:
#################################################################
# CONTROL PROBE: force NEVER varies (both agents locked at one
# value the whole run) -- isolates whether slow convergence is a
# general warm-up issue (zero-init weights + low LR + hard task)
# or specifically caused by block-switching / force variation.
#################################################################

CONTROL_FORCE = 500   # both agents fixed at this force for the entire run

def run_short_probe_no_variation(cost_scale_multiplier, n_steps=SWEEP_STEPS, seed=None,
                                   entropy_coef=None, control_force=CONTROL_FORCE):
    if seed is not None:
        np.random.seed(seed)
        torch.manual_seed(seed)

    entropy_coef_used = entropy_coef if entropy_coef is not None else ENTROPY_COEF
    global FORCE_TO_COST_SCALE
    FORCE_TO_COST_SCALE = BASE_SCALE * cost_scale_multiplier

    # block_len larger than n_steps -- _set_new_block() only runs ONCE, at construction,
    # and never triggers again during the whole run (no block-switching at all).
    env = CurriculumEnv(block_len=n_steps + 1000, lock_fixed_agent='B')
    env.set_training_phase("COOP_FULL_RANDOM", sync_window=TARGET_WINDOW)
    env.gaze_cost = GAZE_COST

    # Override the initial (block-based) force assignment -- BOTH agents locked
    # at the same control_force, bypassing the fixed/varying role logic entirely.
    env.force_A = control_force
    env.force_B = control_force
    env.cost_A = force_to_cost(control_force)
    env.cost_B = force_to_cost(control_force)
    env.cond_label = f"control_fixed_{control_force}g"

    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_A = load_and_expand_agent(agent_A, f"{OLD_CHECKPOINT_DIR}/agent_A_{CHECKPOINT_SET}.pth")
    agent_B = load_and_expand_agent(agent_B, f"{OLD_CHECKPOINT_DIR}/agent_B_{CHECKPOINT_SET}.pth")

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state_with_effort(state_A_dict, env.force_A, env.force_B)
    public_B = preprocess_state_with_effort(state_B_dict, env.force_B, env.force_A)
    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)

    log = []

    for update_step in range(n_steps // N_STEPS):
        mem_pub_A, mem_priv_A, mem_mask_A = [], [], []
        mem_act_A, mem_val_A, mem_rew_A, mem_logp_A = [], [], [], []
        mem_pub_B, mem_priv_B, mem_mask_B = [], [], []
        mem_act_B, mem_val_B, mem_rew_B, mem_logp_B = [], [], [], []

        start_hidden_A, start_hidden_B = hidden_A.detach(), hidden_B.detach()
        hidden_A, hidden_B = start_hidden_A, start_hidden_B

        for _ in range(N_STEPS):
            mask_A = torch.tensor([[0., 0. if state_A_dict['can_pull'] else -1e9]], device=device).float()
            mask_B = torch.tensor([[0., 0. if state_B_dict['can_pull'] else -1e9]], device=device).float()

            mem_pub_A.append(public_A); mem_priv_A.append(private_A); mem_mask_A.append(mask_A)
            mem_pub_B.append(public_B); mem_priv_B.append(private_B); mem_mask_B.append(mask_B)

            with torch.no_grad():
                logits_A, value_A, hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
                logits_B, value_B, hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)

            act_A, log_prob_A = sample_actions(logits_A)
            act_B, log_prob_B = sample_actions(logits_B)
            mem_act_A.append(act_A); mem_val_A.append(value_A); mem_logp_A.append(log_prob_A)
            mem_act_B.append(act_B); mem_val_B.append(value_B); mem_logp_B.append(log_prob_B)

            can_pull_A_now, can_pull_B_now = state_A_dict['can_pull'], state_B_dict['can_pull']
            next_A, next_B, r_A, r_B, done = env.step(act_A, act_B)
            mem_rew_A.append(r_A); mem_rew_B.append(r_B)

            log.append({'A_pull': act_A['pull'], 'B_pull': act_B['pull'],
                        'A_gaze': act_A['gaze'], 'B_gaze': act_B['gaze'],
                        'can_pull_A': can_pull_A_now, 'can_pull_B': can_pull_B_now})

            private_A = get_private_state(next_B, env.B_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_A['gaze']==1)
            private_B = get_private_state(next_A, env.A_time_since_pull, env.current_sync_window, FIXED_RELIABILITY, act_B['gaze']==1)
            public_A = preprocess_state_with_effort(next_A, env.force_A, env.force_B)
            public_B = preprocess_state_with_effort(next_B, env.force_B, env.force_A)
            state_A_dict, state_B_dict = next_A, next_B

        with torch.no_grad():
            _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
            _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)
        returns_A = calculate_gae(mem_rew_A, mem_val_A, next_value_A)
        returns_B = calculate_gae(mem_rew_B, mem_val_B, next_value_B)
        values_A_old = torch.cat(mem_val_A).view(-1)
        values_B_old = torch.cat(mem_val_B).view(-1)
        adv_A = (returns_A - values_A_old); adv_A = (adv_A - adv_A.mean()) / (adv_A.std() + 1e-8)
        adv_B = (returns_B - values_B_old); adv_B = (adv_B - adv_B.mean()) / (adv_B.std() + 1e-8)

        curr_hidden_A, curr_hidden_B = start_hidden_A, start_hidden_B
        new_lp_A_list, new_lp_B_list = [], []
        entropy_A_list, entropy_B_list = [], []
        for t in range(N_STEPS):
            (l_A), _, curr_hidden_A = agent_A(mem_pub_A[t], mem_priv_A[t], curr_hidden_A, mem_mask_A[t])
            m,p,g = l_A; a = mem_act_A[t]
            new_lp_A_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_A = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()
            entropy_A_list.append(ent_A)

            (l_B), _, curr_hidden_B = agent_B(mem_pub_B[t], mem_priv_B[t], curr_hidden_B, mem_mask_B[t])
            m,p,g = l_B; a = mem_act_B[t]
            new_lp_B_list.append(Categorical(logits=m).log_prob(torch.tensor(a['move']).to(device)) +
                                  Categorical(logits=p).log_prob(torch.tensor(a['pull']).to(device)) +
                                  Categorical(logits=g).log_prob(torch.tensor(a['gaze']).to(device)))
            ent_B = -(F.softmax(p, -1)*F.log_softmax(p, -1)).sum()
            entropy_B_list.append(ent_B)

        new_lp_A = torch.stack(new_lp_A_list).view(-1); old_lp_A = torch.cat(mem_logp_A).detach().view(-1)
        new_lp_B = torch.stack(new_lp_B_list).view(-1); old_lp_B = torch.cat(mem_logp_B).detach().view(-1)
        mean_ent_A = torch.stack(entropy_A_list).mean()
        mean_ent_B = torch.stack(entropy_B_list).mean()

        ratio_A = torch.exp(new_lp_A - old_lp_A)
        loss_A = -torch.min(ratio_A*adv_A, torch.clamp(ratio_A,1-EPS_CLIP,1+EPS_CLIP)*adv_A).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_A_old, returns_A) - entropy_coef_used * mean_ent_A
        ratio_B = torch.exp(new_lp_B - old_lp_B)
        loss_B = -torch.min(ratio_B*adv_B, torch.clamp(ratio_B,1-EPS_CLIP,1+EPS_CLIP)*adv_B).mean() + \
                 VALUE_LOSS_COEF*F.mse_loss(values_B_old, returns_B) - entropy_coef_used * mean_ent_B

        optimizer_A.zero_grad(); loss_A.backward(); torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 0.5); optimizer_A.step()
        optimizer_B.zero_grad(); loss_B.backward(); torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 0.5); optimizer_B.step()

        if update_step % (5000 // N_STEPS) == 0:
            steps_so_far = update_step * N_STEPS
            print(f"    [CONTROL x{cost_scale_multiplier} seed={seed}, force={control_force}g] "
                  f"step {steps_so_far}/{n_steps}, R_A: {sum(mem_rew_A):.2f}, R_B: {sum(mem_rew_B):.2f}")

    log_df = pd.DataFrame(log)
    return log_df


def run_control_probe(cost_scale_multiplier=1.0, n_steps=SWEEP_STEPS, seeds=(0,1,2),
                       control_force=CONTROL_FORCE, force_retrain=False):
    out_path = f"control_probe_rawlog_{CHECKPOINT_SET}_force{control_force}_x{cost_scale_multiplier}.pkl"

    if not force_retrain and os.path.exists(out_path):
        print(f"Found existing {out_path} -- loading instead of re-running.")
        return pd.read_pickle(out_path)

    all_logs = []
    for seed in seeds:
        print(f"\n=== CONTROL probe, seed {seed}, force={control_force}g (fixed, no variation) ===")
        log_df = run_short_probe_no_variation(cost_scale_multiplier, n_steps=n_steps, seed=seed,
                                                control_force=control_force)
        log_df['seed'] = seed
        all_logs.append(log_df)

    combined = pd.concat(all_logs, ignore_index=False)
    combined.to_pickle(out_path)
    print(f"\nSaved {out_path}")
    return combined


# --- Run it ---
# control_log = run_control_probe(cost_scale_multiplier=1.0, n_steps=SWEEP_STEPS, force_retrain=True)
run_control_probe(cost_scale_multiplier=1.0, n_steps=SWEEP_STEPS)


In [ ]:
#################################################################
# Compare convergence: control (no variation) vs fixed-role (varying force)
#################################################################
if 0:
    
    STEP_BIN_SIZE = 5000

    fig, ax = plt.subplots(figsize=(12, 7))

    # --- Control curve ---
    per_seed_binned_control = []
    for seed in (0, 1, 2):
        seed_df = control_log[control_log['seed'] == seed].reset_index(drop=True)
        eligible = seed_df[seed_df['can_pull_A']].copy()
        eligible['step_bin'] = (eligible.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
        per_seed_binned_control.append(eligible.groupby('step_bin')['A_pull'].mean())

    binned_df = pd.concat(per_seed_binned_control, axis=1)
    mean_curve, std_curve = binned_df.mean(axis=1), binned_df.std(axis=1)
    line, = ax.plot(mean_curve.index, mean_curve.values, 'o-', label='CONTROL (force fixed, no variation)', linewidth=2.5, color='black')
    ax.fill_between(mean_curve.index, mean_curve.values - std_curve.values,
                     mean_curve.values + std_curve.values, alpha=0.15, color='black')

    # --- Overlay the x1.0 fixed-role (varying force) curve for comparison ---
    per_seed_binned_varying = []
    for seed in FIXEDROLE_SEED_LIST:
        fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x1.0_seed{seed}.pkl"
        try:
            log_df = pd.read_pickle(fname)
        except FileNotFoundError:
            continue
        eligible = log_df[log_df['can_pull_A']].copy()
        eligible['step_bin'] = (eligible.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
        per_seed_binned_varying.append(eligible.groupby('step_bin')['A_pull'].mean())

    if per_seed_binned_varying:
        binned_df2 = pd.concat(per_seed_binned_varying, axis=1)
        mean_curve2, std_curve2 = binned_df2.mean(axis=1), binned_df2.std(axis=1)
        line2, = ax.plot(mean_curve2.index, mean_curve2.values, 'o-', label='VARYING (block-switching, x1.0)', linewidth=2.5, color='orange')
        ax.fill_between(mean_curve2.index, mean_curve2.values - std_curve2.values,
                         mean_curve2.values + std_curve2.values, alpha=0.15, color='orange')

    ax.set_xlabel("Training step")
    ax.set_ylabel("P(pull | eligible)")
    ax.set_title(f"Convergence speed: control (fixed force) vs varying force ({CHECKPOINT_SET})")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"control_vs_varying_convergence_{CHECKPOINT_SET}.png", dpi=150)
    print(f"Saved control_vs_varying_convergence_{CHECKPOINT_SET}.png")

In [ ]:
# Fixed at LOW cost (100g, near-zero effort) the entire run
# run_control_probe(cost_scale_multiplier=1.0, seeds=(0,1,2), control_force=100, force_retrain=False)

# Fixed at HIGH cost (900g, maximum effort) the entire run  
# run_control_probe(cost_scale_multiplier=25.0, seeds=(0,1,2), control_force=900, force_retrain=False)

In [ ]:
#################################################################
# Extended run: 
# the existing run_short_probe_fixed_role, saved with the SAME
# filename convention as the original sweep, so all your existing
# analysis code (efficiency check, plateau check, attempt-rate/IPI
# plots) works on this data without any modification.
#################################################################

EXTENDED_MULT = 18.0
EXTENDED_STEPS = 1000000
EXTENDED_SEEDS = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)

def run_extended_probe(multiplier=EXTENDED_MULT, n_steps=EXTENDED_STEPS,
                        seeds=EXTENDED_SEEDS, block_len=1200, force_retrain=False):
    results = []
    for seed in seeds:
        out_fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{multiplier}_seed{seed}.pkl"

        if not force_retrain and os.path.exists(out_fname):
            print(f"Found existing {out_fname} -- loading instead of re-running.")
            log_df = pd.read_pickle(out_fname)
        else:
            print(f"\n=== EXTENDED run: multiplier=x{multiplier}, seed={seed}, {n_steps} steps ===")
            # n_steps passed EXPLICITLY here (not relying on the function's default),
            # so this works correctly even without a kernel restart.
            corr_A, corr_B, atr_A, atr_B, ent_A, ent_B, n_blocks, log_df = \
                run_short_probe_fixed_role(multiplier, n_steps=n_steps, seed=seed,
                                             block_len=block_len, lock_fixed_agent='B')
            log_df.to_pickle(out_fname)
            print(f"Saved {out_fname}")

        eligible = log_df[log_df['can_pull_A']]
        attempt_rate = eligible['A_pull'].mean() if len(eligible) > 0 else float('nan')
        results.append({'seed': seed, 'attempt_rate': attempt_rate, 'n_steps': len(log_df)})

    results_df = pd.DataFrame(results)
    print(f"\n--- Summary: x{multiplier}, {n_steps} steps ---")
    print(results_df.to_string(index=False))
    print(f"\nMean attempt rate: {results_df['attempt_rate'].mean():.4f} +/- {results_df['attempt_rate'].std():.4f}")
    return results_df


# --- Run it ---
# extended_results = run_extended_probe(force_retrain=True)
run_extended_probe()

In [ ]:
#################################################################
# EXTENDED RUN CHECK: x12.0, 800k steps -- confirms convergence,
# then analyzes attempt rate / efficiency / IPI / gaze vs force
# WITHIN the converged (late-training) window specifically.
#################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

EXT_MULT = 18.0
# EXT_SEEDS = (0, 1, 2)
# EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6)
EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
STEP_BIN_SIZE = 20000
LATE_FRAC = 0.2   # last 20% of training = "converged" window

def reconstruct_successes(log_df, sync_window=10):
    a_events = log_df.index[(log_df['A_pull'] == 1) & (log_df['can_pull_A'])].to_numpy()
    b_events = log_df.index[(log_df['B_pull'] == 1) & (log_df['can_pull_B'])].to_numpy()
    if len(a_events) == 0 or len(b_events) == 0:
        return np.array([]), a_events, b_events
    successes, used_b = [], set()
    for a_idx in a_events:
        candidates = [c for c in b_events if abs(c - a_idx) <= sync_window and c not in used_b]
        if candidates:
            closest = min(candidates, key=lambda c: abs(c - a_idx))
            used_b.add(closest)
            successes.append(a_idx)
    return np.array(successes), a_events, b_events


# --- Load all seeds ---
logs = {}
for seed in EXT_SEEDS:
    fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{EXT_MULT}_seed{seed}.pkl"
    try:
        logs[seed] = pd.read_pickle(fname)
    except FileNotFoundError:
        print(f"Missing {fname}")

print(f"Loaded {len(logs)} seeds for x{EXT_MULT}.\n")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

#################################################################
# PANEL 1: Plateau check -- has it actually converged by 800k?
#################################################################
per_seed_binned = []
for seed, log_df in logs.items():
    eligible = log_df[log_df['can_pull_A']].copy()
    eligible['step_bin'] = (eligible.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
    per_seed_binned.append(eligible.groupby('step_bin')['A_pull'].mean())

binned_df = pd.concat(per_seed_binned, axis=1)
mean_curve, std_curve = binned_df.mean(axis=1), binned_df.std(axis=1)
axes[0,0].plot(mean_curve.index, mean_curve.values, 'o-', color='#2E86AB')
axes[0,0].fill_between(mean_curve.index, mean_curve.values - std_curve.values,
                        mean_curve.values + std_curve.values, alpha=0.2, color='#2E86AB')
axes[0,0].set_xlabel("Training step")
axes[0,0].set_ylabel("P(pull | eligible)")
axes[0,0].set_title(f"Plateau check: x{EXT_MULT}, 800k steps")

# Early vs late comparison, printed
early_rates, late_rates = [], []
for seed, log_df in logs.items():
    n = len(log_df)
    cutoff = int(n * (1 - LATE_FRAC))
    early = log_df.iloc[:cutoff]; late = log_df.iloc[cutoff:]
    early_elig = early[early['can_pull_A']]; late_elig = late[late['can_pull_A']]
    if len(early_elig) > 0: early_rates.append(early_elig['A_pull'].mean())
    if len(late_elig) > 0: late_rates.append(late_elig['A_pull'].mean())

print(f"Early ({int((1-LATE_FRAC)*100)}%) attempt rate: {np.mean(early_rates):.4f} +/- {np.std(early_rates):.4f}")
print(f"Late (last {int(LATE_FRAC*100)}%) attempt rate:  {np.mean(late_rates):.4f} +/- {np.std(late_rates):.4f}")
still_moving = abs(np.mean(late_rates) - np.mean(early_rates))
print(f"Change: {still_moving:.4f} -- {'still moving noticeably' if still_moving > 0.03 else 'looks settled'}\n")

#################################################################
# PANEL 2: Attempt rate vs FORCE, in the LATE (converged) window only
#################################################################
per_seed_binned_force = []
for seed, log_df in logs.items():
    n = len(log_df)
    cutoff = int(n * (1 - LATE_FRAC))
    late = log_df.iloc[cutoff:].copy()
    late_elig = late[late['can_pull_A']].copy()
    if len(late_elig) == 0:
        continue
    late_elig['force_bin'] = (late_elig['force_A'] // 100) * 100
    per_seed_binned_force.append(late_elig.groupby('force_bin')['A_pull'].mean())

if per_seed_binned_force:
    binned_force_df = pd.concat(per_seed_binned_force, axis=1)
    mean_force_curve, std_force_curve = binned_force_df.mean(axis=1), binned_force_df.std(axis=1)
    axes[0,1].errorbar(mean_force_curve.index, mean_force_curve.values, yerr=std_force_curve.values,
                        fmt='o-', color='#C73E1D', capsize=4)
    axes[0,1].set_xlabel("Force (g)")
    axes[0,1].set_ylabel("P(pull | eligible)")
    axes[0,1].set_title(f"CONVERGED attempt rate vs force (last {int(LATE_FRAC*100)}% of training)")

    # Print the actual within-run correlation for the late window
    from scipy import stats as scipy_stats
    all_force_late, all_pull_late = [], []
    for seed, log_df in logs.items():
        n = len(log_df); cutoff = int(n * (1 - LATE_FRAC))
        late_elig = log_df.iloc[cutoff:]
        late_elig = late_elig[late_elig['can_pull_A']]
        all_force_late.extend(late_elig['force_A'].tolist())
        all_pull_late.extend(late_elig['A_pull'].tolist())
    slope, intercept, r, p, se = scipy_stats.linregress(all_force_late, all_pull_late)
    print(f"CONVERGED-window regression: slope={slope*100:.5f} per 100g, p={p:.4f}, "
          f"n={len(all_force_late)}, direction={'predicted (negative)' if slope<0 else 'reversed (positive)'}")

#################################################################
# PANEL 3: Efficiency (success/attempt), early vs late
#################################################################
early_effs, late_effs = [], []
early_n, late_n = [], []
for seed, log_df in logs.items():
    n = len(log_df); cutoff = int(n * (1 - LATE_FRAC))
    early_df = log_df.iloc[:cutoff]; late_df = log_df.iloc[cutoff:]
    for df, eff_list, n_list in [(early_df, early_effs, early_n), (late_df, late_effs, late_n)]:
        successes, a_events, b_events = reconstruct_successes(df, sync_window=TARGET_WINDOW)
        n_list.append(len(a_events))
        if len(a_events) > 0:
            eff_list.append(len(successes) / len(a_events))

axes[1,0].bar(['Early', 'Late (converged)'],
              [np.mean(early_effs) if early_effs else 0, np.mean(late_effs) if late_effs else 0],
              yerr=[np.std(early_effs) if early_effs else 0, np.std(late_effs) if late_effs else 0],
              color=['#888888', '#2E86AB'], capsize=8)
axes[1,0].set_ylabel("Efficiency (success / attempt)")
axes[1,0].set_ylim(0, 1.0)
axes[1,0].set_title("Efficiency: early vs converged window")
print(f"Early efficiency: {np.mean(early_effs):.4f} +/- {np.std(early_effs):.4f} (n_attempts/seed: {early_n})")
print(f"Late efficiency:  {np.mean(late_effs):.4f} +/- {np.std(late_effs):.4f} (n_attempts/seed: {late_n})")

#################################################################
# PANEL 4: Gaze rate vs force, late window
#################################################################
per_seed_binned_gaze = []
for seed, log_df in logs.items():
    if 'A_gaze' not in log_df.columns:
        continue
    n = len(log_df); cutoff = int(n * (1 - LATE_FRAC))
    late = log_df.iloc[cutoff:].copy()
    late['force_bin'] = (late['force_A'] // 100) * 100
    per_seed_binned_gaze.append(late.groupby('force_bin')['A_gaze'].mean())

if per_seed_binned_gaze:
    binned_gaze_df = pd.concat(per_seed_binned_gaze, axis=1)
    mean_gaze_curve, std_gaze_curve = binned_gaze_df.mean(axis=1), binned_gaze_df.std(axis=1)
    axes[1,1].errorbar(mean_gaze_curve.index, mean_gaze_curve.values, yerr=std_gaze_curve.values,
                        fmt='o-', color='#6A4C93', capsize=4)
    axes[1,1].set_xlabel("Force (g)")
    axes[1,1].set_ylabel("P(gaze=1)")
    axes[1,1].set_title(f"CONVERGED gaze rate vs force (last {int(LATE_FRAC*100)}%)")

plt.tight_layout()
plt.savefig(f"extended_x{EXT_MULT}_full_check_{CHECKPOINT_SET}.png", dpi=150)
print(f"\nSaved extended_x{EXT_MULT}_full_check_{CHECKPOINT_SET}.png")

In [ ]:
#################################################################
# Look closer: per-seed curves (not just the averaged band), plus
# check whether Agent B was still co-adapting during the same
# window, and whether reward tracked the "plateau then rise" shape.
#################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

EXT_MULT = 18.0
# EXT_SEEDS = (0, 1, 2)
# EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6)
EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
STEP_BIN_SIZE = 20000

logs = {}
for seed in EXT_SEEDS:
    fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{EXT_MULT}_seed{seed}.pkl"
    try:
        logs[seed] = pd.read_pickle(fname)
    except FileNotFoundError:
        print(f"Missing {fname}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: EACH SEED separately, not averaged ---
colors = ['#2E86AB', '#C73E1D', '#4CAF50', '#F5A623', '#8E44AD',
          '#16A085', '#E91E63', '#795548', '#607D8B', '#F1C40F']
for i, (seed, log_df) in enumerate(logs.items()):
    eligible = log_df[log_df['can_pull_A']].copy()
    eligible['step_bin'] = (eligible.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
    curve = eligible.groupby('step_bin')['A_pull'].mean()
    axes[0].plot(curve.index, curve.values, 'o-', label=f'seed {seed}', color=colors[i], alpha=0.8)

axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Agent A: P(pull | eligible)")
axes[0].set_title("Per-seed curves (not averaged) -- is 'plateau then rise' shared or one seed's artifact?")
axes[0].legend()

# --- Panel 2: Agent B's behavior over the SAME window (co-adaptation check) ---
for i, (seed, log_df) in enumerate(logs.items()):
    eligible_B = log_df[log_df['can_pull_B']].copy()
    eligible_B['step_bin'] = (eligible_B.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
    curve_B = eligible_B.groupby('step_bin')['B_pull'].mean()
    axes[1].plot(curve_B.index, curve_B.values, 'o-', label=f'seed {seed}', color=colors[i], alpha=0.8)

axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Agent B: P(pull | eligible)")
axes[1].set_title("Agent B's own behavior over the same window\n(still moving = co-adaptation, not a stable opponent)")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"per_seed_and_B_check_x{EXT_MULT}_{CHECKPOINT_SET}.png", dpi=150)
print(f"Saved per_seed_and_B_check_x{EXT_MULT}_{CHECKPOINT_SET}.png")

# --- Reward trend over the same window, per seed ---
fig2, ax2 = plt.subplots(figsize=(10, 6))
for i, (seed, log_df) in enumerate(logs.items()):
    if 'A_reward' not in log_df.columns:
        print(f"(seed {seed}: no per-step reward logged in this raw log -- skipping reward panel)")
        continue
    log_df_copy = log_df.copy()
    log_df_copy['step_bin'] = (log_df_copy.index // STEP_BIN_SIZE) * STEP_BIN_SIZE
    reward_curve = log_df_copy.groupby('step_bin')['A_reward'].mean()
    ax2.plot(reward_curve.index, reward_curve.values, 'o-', label=f'seed {seed}', color=colors[i])
ax2.set_xlabel("Training step")
ax2.set_ylabel("Mean A_reward per step")
ax2.set_title("Does reward track the plateau/rise shape? (real improvement vs. drift without benefit)")
ax2.legend()
plt.tight_layout()
plt.savefig(f"reward_trend_x{EXT_MULT}_{CHECKPOINT_SET}.png", dpi=150)
print(f"Saved reward_trend_x{EXT_MULT}_{CHECKPOINT_SET}.png")

In [ ]:
#################################################################
# Objective stability classification: is each seed's LATE-training
# slope actually distinguishable from zero, or still trending?
#################################################################
import pandas as pd
import numpy as np
from scipy import stats

EXT_MULT = 18.0
# EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6)
EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)
TAIL_FRAC = 0.25   # check the last 30% of training for a residual trend

print(f"{'Seed':>6} | {'Tail slope (per 100k steps)':>28} | {'p-value':>10} | {'Classification':>16}")

classifications = {}
for seed in EXT_SEEDS:
    fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{EXT_MULT}_seed{seed}.pkl"
    try:
        log_df = pd.read_pickle(fname)
    except FileNotFoundError:
        continue

    n = len(log_df)
    tail = log_df.iloc[int(n * (1 - TAIL_FRAC)):]
    tail_elig = tail[tail['can_pull_A']].copy()

    if len(tail_elig) < 50:
        print(f"{seed:>6} | NOT ENOUGH DATA")
        continue

    slope, intercept, r, p, se = stats.linregress(tail_elig.index.values, tail_elig['A_pull'].values)
    slope_per_100k = slope * 100000

    is_stable = p >= 0.05   # can't reject "slope is zero"
    classifications[seed] = 'stable' if is_stable else 'climbing'
    print(f"{seed:>6} | {slope_per_100k:>27.4f} | {p:>10.4f} | {classifications[seed]:>16}")

print(f"\nStable seeds: {[s for s,c in classifications.items() if c=='stable']}")
print(f"Climbing seeds: {[s for s,c in classifications.items() if c=='climbing']}")

In [ ]:
#################################################################
# Pull rate, gaze rate, IPI, and success rate vs FORCE -- using
# the same LATE_FRAC window for every seed (not adaptive per-seed).
# Combined by averaging per-seed binned curves (seed-first, not
# raw pooling).
#################################################################
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

EXT_MULT = 20.0
EXT_SEEDS = (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)   # the seeds you've selected as "stable"
LATE_FRAC = 0.4   # <-- change this: fraction of the run (from the end) treated as "converged"


def reconstruct_successes_labeled(log_df, sync_window=10):
    """
    Same matching logic as before, but also returns a boolean array
    (same length as a_events) indicating which of A's attempts succeeded --
    needed to bin success rate BY FORCE, not just get one overall number.
    """
    a_events = log_df.index[(log_df['A_pull'] == 1) & (log_df['can_pull_A'])].to_numpy()
    b_events = log_df.index[(log_df['B_pull'] == 1) & (log_df['can_pull_B'])].to_numpy()
    a_success = np.zeros(len(a_events), dtype=bool)
    if len(a_events) == 0 or len(b_events) == 0:
        return a_events, a_success
    used_b = set()
    for i, a_idx in enumerate(a_events):
        candidates = [c for c in b_events if abs(c - a_idx) <= sync_window and c not in used_b]
        if candidates:
            closest = min(candidates, key=lambda c: abs(c - a_idx))
            used_b.add(closest)
            a_success[i] = True
    return a_events, a_success


# --- Load logs ---
logs = {}
for seed in EXT_SEEDS:
    fname = f"cost_scale_sweep_fixedrole_rawlog_{CHECKPOINT_SET}_x{EXT_MULT}_seed{seed}.pkl"
    try:
        logs[seed] = pd.read_pickle(fname)
    except FileNotFoundError:
        print(f"Missing {fname}")

print(f"Loaded {len(logs)} seeds for x{EXT_MULT}: {list(logs.keys())}")
print(f"Using LATE_FRAC={LATE_FRAC} -- last {LATE_FRAC*100:.0f}% of each run\n")

# --- Per-seed, per-force-bin curves for each metric ---
per_seed_pull_A, per_seed_pull_B = [], []
per_seed_gaze_A, per_seed_gaze_B = [], []
per_seed_ipi_A, per_seed_ipi_B = [], []
per_seed_success = []

for seed, log_df in logs.items():
    n = len(log_df)
    late = log_df.iloc[int(n * (1 - LATE_FRAC)):].copy()
    late['force_bin'] = (late['force_A'] // 10) * 10

    # --- Pull rate vs force (conditioned on eligibility) ---
    elig_A = late[late['can_pull_A']]
    elig_B = late[late['can_pull_B']]
    per_seed_pull_A.append(elig_A.groupby('force_bin')['A_pull'].mean())
    per_seed_pull_B.append(elig_B.groupby('force_bin')['B_pull'].mean())

    # --- Gaze rate vs force (unconditioned) ---
    if 'A_gaze' in late.columns:
        per_seed_gaze_A.append(late.groupby('force_bin')['A_gaze'].mean())
        per_seed_gaze_B.append(late.groupby('force_bin')['B_gaze'].mean())

    # --- IPI vs force (binned by force at the START of each interval) ---
    for agent, storage in [('A', per_seed_ipi_A), ('B', per_seed_ipi_B)]:
        pull_col, can_pull_col = f'{agent}_pull', f'can_pull_{agent}'
        pull_idx = late.index[(late[pull_col] == 1) & (late[can_pull_col])].to_numpy()
        if len(pull_idx) < 2:
            continue
        ipi = np.diff(pull_idx)
        force_at_start = late.loc[pull_idx[:-1], 'force_A'].to_numpy()  # A's force, since it's the varying one
        ipi_df = pd.DataFrame({'force_bin': (force_at_start // 10) * 10, 'ipi': ipi})
        storage.append(ipi_df.groupby('force_bin')['ipi'].mean())

    # --- Success rate vs force (A's attempts, binned by force at attempt time) ---
    a_events, a_success = reconstruct_successes_labeled(late, sync_window=TARGET_WINDOW)
    if len(a_events) > 0:
        force_at_attempt = late.loc[a_events, 'force_A'].to_numpy()
        succ_df = pd.DataFrame({'force_bin': (force_at_attempt // 10) * 10, 'success': a_success})
        per_seed_success.append(succ_df.groupby('force_bin')['success'].mean())


def combine_and_plot(ax, per_seed_list, title, ylabel, color, ylim=None):
    """Averages per-seed binned curves (seed-first), plots mean +/- std band."""
    if not per_seed_list:
        ax.set_title(title + "\n(no data)")
        return
    combined = pd.concat(per_seed_list, axis=1)
    mean_curve, std_curve = combined.mean(axis=1), combined.std(axis=1)
    ax.plot(mean_curve.index, mean_curve.values, 'o-', color=color)
    ax.fill_between(mean_curve.index, mean_curve.values - std_curve.values,
                     mean_curve.values + std_curve.values, alpha=0.2, color=color)
    ax.set_xlabel("Force (g)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if ylim: ax.set_ylim(*ylim)


fig, axes = plt.subplots(2, 2, figsize=(15, 11))

combine_and_plot(axes[0,0], per_seed_pull_A, "Agent A: attempt rate vs force", "P(pull | eligible)", '#2E86AB', ylim=(0,1.05))
combine_and_plot(axes[0,1], per_seed_gaze_A, "Agent A: gaze rate vs force", "P(gaze=1)", '#6A4C93', ylim=(0,1.05))
combine_and_plot(axes[1,0], per_seed_ipi_A, "Agent A: mean IPI vs force", "Steps", '#F5A623')
combine_and_plot(axes[1,1], per_seed_success, "Cooperation success rate vs force\n(A's force at attempt time)", "Success rate", '#4CAF50', ylim=(0,1.05))

plt.suptitle(f"x{EXT_MULT}, last {LATE_FRAC*100:.0f}% of training, seeds {EXT_SEEDS} (n={len(logs)})", y=1.02)
plt.tight_layout()
plt.savefig(f"vs_force_stable_seeds_x{EXT_MULT}_{CHECKPOINT_SET}.png", dpi=150)
print(f"Saved vs_force_stable_seeds_x{EXT_MULT}_{CHECKPOINT_SET}.png")

# --- Agent B's behavior vs A's (partner's) force, since B is fixed at 100g the whole time ---
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 5.5))
combine_and_plot(axes2[0], per_seed_pull_B, "Agent B: attempt rate vs A's force", "P(pull | eligible)", '#C73E1D', ylim=(0,1.05))
combine_and_plot(axes2[1], per_seed_gaze_B, "Agent B: gaze rate vs A's force", "P(gaze=1)", '#795548', ylim=(0,1.05))
plt.suptitle(f"Agent B's behavior vs A's (partner's) force -- x{EXT_MULT}, last {LATE_FRAC*100:.0f}%", y=1.02)
plt.tight_layout()
plt.savefig(f"agentB_vs_partner_force_x{EXT_MULT}_{CHECKPOINT_SET}.png", dpi=150)
print(f"Saved agentB_vs_partner_force_x{EXT_MULT}_{CHECKPOINT_SET}.png")

### entropy-coefficient sweep; not in use anymore
Fixed at the strongest cost condition already tested (25x), sweeps `entropy_coef` instead -- checking whether the flat attempt-rate result across every other axis (cost scale, checkpoint, cooldown structure) is actually an exploration/optimization problem: an already-confident pretrained policy may need a bigger entropy bonus than the default `ENTROPY_COEF=0.0001` to meaningfully explore "not pulling" often enough to learn whether it's worthwhile.

Tracks REALIZED entropy (not just the coefficient) so we can confirm the knob is actually doing something, rather than assuming it from the coefficient value alone.

In [ ]:
#################################################################
# ENTROPY-COEFFICIENT SWEEP
#################################################################
if 0:
    ENTROPY_SWEEP_VALUES = [0.0001, 0.01, 0.05, 0.1, 0.2]  # 0.0001 = current baseline
    FIXED_MULT_FOR_ENTROPY_TEST = 25.0  # strongest cost condition already tested
    ENTROPY_SWEEP_RESULTS_OUT = f"entropy_sweep_results_{CHECKPOINT_SET}.pkl"


    def run_entropy_sweep(force_retrain=False, seed=0):
        """
        force_retrain=False (default): if results already exist, skip and print them.
        force_retrain=True: always re-run, overwriting existing outputs.
        """
        if not force_retrain and os.path.exists(ENTROPY_SWEEP_RESULTS_OUT):
            print(f"Found existing {ENTROPY_SWEEP_RESULTS_OUT} -- loading instead of re-running.")
            print("Call run_entropy_sweep(force_retrain=True) to redo the sweep.\n")
            results_df = pd.read_pickle(ENTROPY_SWEEP_RESULTS_OUT)
            print(results_df.to_string(index=False))
            return results_df

        if force_retrain and os.path.exists(ENTROPY_SWEEP_RESULTS_OUT):
            print(f"force_retrain=True -- overwriting existing {ENTROPY_SWEEP_RESULTS_OUT}\n")

        print(f"Running entropy sweep at fixed cost multiplier x{FIXED_MULT_FOR_ENTROPY_TEST}, "
              f"checkpoint={CHECKPOINT_SET}, seed={seed}\n")

        results = []
        print(f"{'entropy_coef':>14} | {'realized_ent_A':>15} | {'realized_ent_B':>15} | "
              f"{'attempt_A':>10} | {'attempt_B':>10} | {'corr_A':>8} | {'corr_B':>8}")
        for ecoef in ENTROPY_SWEEP_VALUES:
            corr_A, corr_B, pct_elig_A, pct_elig_B, atr_A, atr_B, ent_A, ent_B, log_df = \
                run_short_probe(FIXED_MULT_FOR_ENTROPY_TEST, seed=seed, entropy_coef=ecoef)
            print(f"{ecoef:>14} | {ent_A:>15.4f} | {ent_B:>15.4f} | "
                  f"{atr_A:>10.4f} | {atr_B:>10.4f} | {corr_A:>8.4f} | {corr_B:>8.4f}")
            results.append({
                'entropy_coef': ecoef, 'realized_entropy_A': ent_A, 'realized_entropy_B': ent_B,
                'attempt_rate_A': atr_A, 'attempt_rate_B': atr_B, 'corr_A': corr_A, 'corr_B': corr_B,
            })
            log_df.to_pickle(f"entropy_sweep_rawlog_{CHECKPOINT_SET}_ecoef{ecoef}.pkl")

        results_df = pd.DataFrame(results)
        results_df.to_pickle(ENTROPY_SWEEP_RESULTS_OUT)
        print(f"\nSaved {ENTROPY_SWEEP_RESULTS_OUT}")
        print("\nCheck: does realized_ent_A/B actually climb as entropy_coef increases?")
        print("If yes, the knob is working -- then check whether attempt_A/B finally")
        print("moves away from its ~0.30 / ~1.0 plateau. If realized entropy climbs but")
        print("attempt rate still doesn't move, that's a more surprising result worth")
        print("treating carefully rather than just reaching for an even bigger coefficient.")
        return results_df


    # --- To run the entropy sweep ---
    # run_entropy_sweep()                    # skips if results already exist
    # run_entropy_sweep(force_retrain=True)  # always re-runs
    entropy_results_df = run_entropy_sweep()


In [ ]:
"""
Entropy-sweep summary and diagnostic: is the entropy bonus actually reaching
the PULL decision, or is the network satisfying it cheaply via gaze noise?

Loads the aggregated entropy_sweep_results pkl -- no retraining needed.
"""
if 0:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    CHECKPOINT_SET = "generalist_withGAE_PPO"
    results_df = pd.read_pickle(f"entropy_sweep_results_{CHECKPOINT_SET}.pkl")
    print("Raw results:")
    print(results_df.to_string(index=False))

    # --- Compute a PULL-ONLY entropy proxy from the observed attempt rate ---
    # If pull were a simple Bernoulli(p) with p = attempt_rate, its entropy would be:
    def binary_entropy(p, eps=1e-9):
        p = np.clip(p, eps, 1 - eps)
        return -p * np.log(p) - (1 - p) * np.log(1 - p)

    results_df['pull_only_entropy_A'] = binary_entropy(results_df['attempt_rate_A'])
    results_df['pull_only_entropy_B'] = binary_entropy(results_df['attempt_rate_B'])

    # --- The gap: how much of the REPORTED joint entropy is NOT explained by pull alone ---
    results_df['entropy_gap_A'] = results_df['realized_entropy_A'] - results_df['pull_only_entropy_A']
    results_df['entropy_gap_B'] = results_df['realized_entropy_B'] - results_df['pull_only_entropy_B']

    print("\n--- Pull-only entropy vs. reported joint entropy ---")
    print(results_df[['entropy_coef', 'realized_entropy_A', 'pull_only_entropy_A', 'entropy_gap_A',
                       'realized_entropy_B', 'pull_only_entropy_B', 'entropy_gap_B']].to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(results_df['entropy_coef'], results_df['realized_entropy_A'], 'o-', label='Agent A: joint (reported)')
    axes[0].plot(results_df['entropy_coef'], results_df['pull_only_entropy_A'], 's--', label='Agent A: pull-only (implied)')
    axes[0].plot(results_df['entropy_coef'], results_df['realized_entropy_B'], 'o-', label='Agent B: joint (reported)')
    axes[0].plot(results_df['entropy_coef'], results_df['pull_only_entropy_B'], 's--', label='Agent B: pull-only (implied)')
    axes[0].set_xlabel('entropy_coef')
    axes[0].set_ylabel('Entropy (nats)')
    axes[0].set_title('Reported (joint) vs. pull-only entropy')
    axes[0].legend(fontsize=8)

    axes[1].plot(results_df['entropy_coef'], results_df['attempt_rate_A'], 'o-', label='Agent A')
    axes[1].plot(results_df['entropy_coef'], results_df['attempt_rate_B'], 'o-', label='Agent B')
    axes[1].set_xlabel('entropy_coef')
    axes[1].set_ylabel('Attempt rate (P(pull | eligible))')
    axes[1].set_title('Attempt rate vs. entropy_coef')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend()

    axes[2].plot(results_df['entropy_coef'], results_df['corr_A'], 'o-', label='corr_A (force vs pull)')
    axes[2].plot(results_df['entropy_coef'], results_df['corr_B'], 'o-', label='corr_B (force vs pull)')
    axes[2].axhline(0, color='gray', linewidth=0.8)
    axes[2].set_xlabel('entropy_coef')
    axes[2].set_ylabel('Correlation')
    axes[2].set_title('Cost-sensitivity (force-pull correlation) vs. entropy_coef')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(f'entropy_sweep_diagnostic_{CHECKPOINT_SET}.png', dpi=150)
    print(f"\nSaved entropy_sweep_diagnostic_{CHECKPOINT_SET}.png")

    print("\n--- Interpretation guide ---")
    print("If 'pull-only entropy' stays near zero while 'joint entropy' climbs a lot,")
    print("that's direct evidence the entropy bonus is being satisfied through gaze")
    print("(or move) noise, not the pull decision itself -- meaning entropy_coef needs")
    print("to be applied SEPARATELY and more heavily to just the pull head to actually")
    print("test the entropy-collapse hypothesis on the decision that matters.")

### main training function, with the best cost option and pre-train model

In [ ]:
#################################################################
# 6. EFFORT-INEQUITY FINE-TUNING LOOP
#################################################################

AGENT_A_OUT = "agent_A_effort_withGAE_PPO.pth"
AGENT_B_OUT = "agent_B_effort_withGAE_PPO.pth"
LOG_OUT = "effort_training_log.pkl"


def train_effort_finetune(force_retrain=False):
    """
    force_retrain=False (default): if all three output files already exist,
    skip training entirely and just report what's there -- avoids accidentally
    burning HPC time re-running a finished job.
    force_retrain=True: always retrain, overwriting existing outputs.
    """
    existing = [f for f in (AGENT_A_OUT, AGENT_B_OUT, LOG_OUT) if os.path.exists(f)]

    if not force_retrain and len(existing) == 3:
        print(f"Found existing outputs: {existing}")
        print("Skipping training. Call train_effort_finetune(force_retrain=True) to redo.")
        return
    elif not force_retrain and 0 < len(existing) < 3:
        print(f"WARNING: found a PARTIAL set of existing outputs: {existing}")
        print("This looks like a previous run didn't finish cleanly. "
              "Proceeding with a fresh run (will overwrite).")
    elif force_retrain and existing:
        print(f"force_retrain=True -- overwriting existing outputs: {existing}")

    print("--- Starting Effort-Inequity fine-tune (loaded from gaze efficient_expert) ---\n")

    env = CurriculumEnv(block_len=BLOCK_LEN)
    env.set_training_phase("COOP_FULL_RANDOM", sync_window=TARGET_WINDOW)
    env.gaze_cost = GAZE_COST

    agent_A = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)
    agent_B = AgentBrain(PUBLIC_INPUT_SIZE, PRIVATE_INPUT_SIZE, RNN_HIDDEN_SIZE,
                         MOVE_ACTIONS, PULL_ACTIONS, GAZE_ACTIONS).to(device)

    try:
        agent_A = load_and_expand_agent(agent_A, f"{OLD_CHECKPOINT_DIR}/agent_A_efficient_expert.pth")
        agent_B = load_and_expand_agent(agent_B, f"{OLD_CHECKPOINT_DIR}/agent_B_efficient_expert.pth")
    except FileNotFoundError as e:
        print(f"Error: could not find source checkpoint -- {e}")
        print("Check OLD_CHECKPOINT_DIR in the weight-transfer cell above.")
        return

    optimizer_A = optim.Adam(agent_A.parameters(), lr=LR_FINE_TUNE)
    optimizer_B = optim.Adam(agent_B.parameters(), lr=LR_FINE_TUNE)

    state_A_dict, state_B_dict = env.reset()
    public_A = preprocess_state_with_effort(state_A_dict, env.force_A, env.force_B)
    public_B = preprocess_state_with_effort(state_B_dict, env.force_B, env.force_A)

    private_A = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)
    private_B = torch.zeros(1, PRIVATE_INPUT_SIZE).to(device)

    hidden_A = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)
    hidden_B = torch.zeros(1, RNN_HIDDEN_SIZE).to(device)

    effort_training_log = []

    for update_step in range(CONTINUE_STEPS // N_STEPS):
        total_steps_so_far = update_step * N_STEPS

        mem_pub_A, mem_priv_A, mem_mask_A = [], [], []
        mem_act_A, mem_val_A, mem_rew_A, mem_logp_A = [], [], [], []

        mem_pub_B, mem_priv_B, mem_mask_B = [], [], []
        mem_act_B, mem_val_B, mem_rew_B, mem_logp_B = [], [], [], []

        start_hidden_A = hidden_A.detach()
        start_hidden_B = hidden_B.detach()
        hidden_A = start_hidden_A
        hidden_B = start_hidden_B

        # =================================================================
        # ROLLOUT PHASE
        # =================================================================
        for step_in_rollout in range(N_STEPS):

            if state_A_dict['can_pull']: mask_A = torch.tensor([[0., 0.]], device=device).float()
            else: mask_A = torch.tensor([[0., -1e9]], device=device).float()

            if state_B_dict['can_pull']: mask_B = torch.tensor([[0., 0.]], device=device).float()
            else: mask_B = torch.tensor([[0., -1e9]], device=device).float()

            mem_pub_A.append(public_A); mem_priv_A.append(private_A); mem_mask_A.append(mask_A)
            mem_pub_B.append(public_B); mem_priv_B.append(private_B); mem_mask_B.append(mask_B)

            with torch.no_grad():
                (logits_A_tuple), value_A, hidden_A = agent_A(public_A, private_A, hidden_A, mask_A)
                (logits_B_tuple), value_B, hidden_B = agent_B(public_B, private_B, hidden_B, mask_B)

            act_A_dict, log_prob_A = sample_actions(logits_A_tuple)
            act_B_dict, log_prob_B = sample_actions(logits_B_tuple)

            mem_act_A.append(act_A_dict); mem_val_A.append(value_A); mem_logp_A.append(log_prob_A)
            mem_act_B.append(act_B_dict); mem_val_B.append(value_B); mem_logp_B.append(log_prob_B)

            cond_label_this_step = env.cond_label  # capture before env.step() may advance the block

            next_state_A_dict, next_state_B_dict, reward_A, reward_B, done = \
                env.step(act_A_dict, act_B_dict)

            mem_rew_A.append(reward_A)
            mem_rew_B.append(reward_B)

            effort_training_log.append({
                'step': total_steps_so_far + step_in_rollout,
                'contingency': cond_label_this_step,
                'cost_A': env.cost_A, 'cost_B': env.cost_B,
                'A_pull': act_A_dict['pull'], 'B_pull': act_B_dict['pull'],
                'A_gaze': act_A_dict['gaze'], 'B_gaze': act_B_dict['gaze'],
                'A_reward': reward_A, 'B_reward': reward_B
            })

            sensor_A_active = (act_A_dict['gaze'] == 1)
            sensor_B_active = (act_B_dict['gaze'] == 1)

            private_A = get_private_state(next_state_B_dict, env.B_time_since_pull,
                                           env.current_sync_window, FIXED_RELIABILITY, sensor_A_active)
            private_B = get_private_state(next_state_A_dict, env.A_time_since_pull,
                                           env.current_sync_window, FIXED_RELIABILITY, sensor_B_active)

            public_A = preprocess_state_with_effort(next_state_A_dict, env.force_A, env.force_B)
            public_B = preprocess_state_with_effort(next_state_B_dict, env.force_B, env.force_A)
            state_A_dict = next_state_A_dict
            state_B_dict = next_state_B_dict

        # =================================================================
        # UPDATE PHASE (PPO fix -- same structure as the gaze model)
        # =================================================================
        with torch.no_grad():
            _, next_value_A, _ = agent_A(public_A, private_A, hidden_A, mask_A)
            _, next_value_B, _ = agent_B(public_B, private_B, hidden_B, mask_B)

        returns_A = calculate_gae(mem_rew_A, mem_val_A, next_value_A)
        returns_B = calculate_gae(mem_rew_B, mem_val_B, next_value_B)

        values_A_old = torch.cat(mem_val_A).view(-1)
        values_B_old = torch.cat(mem_val_B).view(-1)

        adv_A = returns_A - values_A_old
        adv_B = returns_B - values_B_old
        adv_A = (adv_A - adv_A.mean()) / (adv_A.std() + 1e-8)
        adv_B = (adv_B - adv_B.mean()) / (adv_B.std() + 1e-8)

        curr_hidden_A = start_hidden_A
        curr_hidden_B = start_hidden_B

        new_logprobs_A_list, entropy_A_list = [], []
        new_logprobs_B_list, entropy_B_list = [], []

        for t in range(N_STEPS):
            (logits_A), _, curr_hidden_A = agent_A(mem_pub_A[t], mem_priv_A[t], curr_hidden_A, mem_mask_A[t])
            move_l, pull_l, gaze_l = logits_A
            taken_act = mem_act_A[t]
            new_lp = Categorical(logits=move_l).log_prob(torch.tensor(taken_act['move']).to(device)) + \
                     Categorical(logits=pull_l).log_prob(torch.tensor(taken_act['pull']).to(device)) + \
                     Categorical(logits=gaze_l).log_prob(torch.tensor(taken_act['gaze']).to(device))
            new_logprobs_A_list.append(new_lp)
            ent = -(F.softmax(pull_l, -1)*F.log_softmax(pull_l, -1)).sum() - \
                   (F.softmax(gaze_l, -1)*F.log_softmax(gaze_l, -1)).sum()
            entropy_A_list.append(ent)

            (logits_B), _, curr_hidden_B = agent_B(mem_pub_B[t], mem_priv_B[t], curr_hidden_B, mem_mask_B[t])
            move_l, pull_l, gaze_l = logits_B
            taken_act = mem_act_B[t]
            new_lp = Categorical(logits=move_l).log_prob(torch.tensor(taken_act['move']).to(device)) + \
                     Categorical(logits=pull_l).log_prob(torch.tensor(taken_act['pull']).to(device)) + \
                     Categorical(logits=gaze_l).log_prob(torch.tensor(taken_act['gaze']).to(device))
            new_logprobs_B_list.append(new_lp)
            ent = -(F.softmax(pull_l, -1)*F.log_softmax(pull_l, -1)).sum() - \
                   (F.softmax(gaze_l, -1)*F.log_softmax(gaze_l, -1)).sum()
            entropy_B_list.append(ent)

        new_logprobs_A = torch.stack(new_logprobs_A_list).view(-1)
        mean_ent_A = torch.stack(entropy_A_list).mean()
        old_logprobs_A = torch.cat(mem_logp_A).detach().view(-1)

        new_logprobs_B = torch.stack(new_logprobs_B_list).view(-1)
        mean_ent_B = torch.stack(entropy_B_list).mean()
        old_logprobs_B = torch.cat(mem_logp_B).detach().view(-1)

        ratio_A = torch.exp(new_logprobs_A - old_logprobs_A)
        surr1_A = ratio_A * adv_A
        surr2_A = torch.clamp(ratio_A, 1-EPS_CLIP, 1+EPS_CLIP) * adv_A
        loss_A = -torch.min(surr1_A, surr2_A).mean() + VALUE_LOSS_COEF * F.mse_loss(values_A_old, returns_A) - ENTROPY_COEF * mean_ent_A

        ratio_B = torch.exp(new_logprobs_B - old_logprobs_B)
        surr1_B = ratio_B * adv_B
        surr2_B = torch.clamp(ratio_B, 1-EPS_CLIP, 1+EPS_CLIP) * adv_B
        loss_B = -torch.min(surr1_B, surr2_B).mean() + VALUE_LOSS_COEF * F.mse_loss(values_B_old, returns_B) - ENTROPY_COEF * mean_ent_B

        optimizer_A.zero_grad(); loss_A.backward(); torch.nn.utils.clip_grad_norm_(agent_A.parameters(), 0.5); optimizer_A.step()
        optimizer_B.zero_grad(); loss_B.backward(); torch.nn.utils.clip_grad_norm_(agent_B.parameters(), 0.5); optimizer_B.step()

        if update_step % (5000 // N_STEPS) == 0:
            print(f"Step: {total_steps_so_far}, Contingency: {env.cond_label}, "
                  f"R_A: {sum(mem_rew_A):.2f}, R_B: {sum(mem_rew_B):.2f}, "
                  f"Ent_A: {mean_ent_A.item():.4f}, Ent_B: {mean_ent_B.item():.4f}")

    print("Effort-inequity fine-tuning complete.")

    df = pd.DataFrame(effort_training_log)
    df.to_pickle(LOG_OUT)
    torch.save(agent_A.state_dict(), AGENT_A_OUT)
    torch.save(agent_B.state_dict(), AGENT_B_OUT)
    print(f"Saved: {AGENT_A_OUT}, {AGENT_B_OUT}, {LOG_OUT}")


# --- To run the training ---
# train_effort_finetune()                    # skips if outputs already exist
# train_effort_finetune(force_retrain=True)  # always retrains, overwrites outputs
if __name__ == "__main__":
    train_effort_finetune()

### do some checking plot

In [ ]:
"""
Checks whether the ~10x A vs B pull-rate gap seen in the effort model
already existed in the pre-effort gaze-generalist model, or emerged during
effort fine-tuning.

Run from the effort-task folder, but pointing back at the gaze-generalist
log too (adjust GENERALIST_LOG_PATH if your relative path differs).
"""
if 0:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    GENERALIST_LOG_PATH = "../MARL_gaze_accumulation_strategies/training_withGAE_PPO_randomPullCoolTime/log_generalist_withGAE_PPO.pkl"
    EFFORT_LOG_PATH = "effort_training_log.pkl"

    # --- 1. Pre-effort baseline: the gaze-generalist model's own pull rates ---
    gen_df = pd.read_pickle(GENERALIST_LOG_PATH)
    pre_A_rate = gen_df['A_pull'].mean()
    pre_B_rate = gen_df['B_pull'].mean()
    print(f"PRE-EFFORT (gaze generalist, full {len(gen_df)} steps):")
    print(f"  Agent A pull rate: {pre_A_rate:.4f}")
    print(f"  Agent B pull rate: {pre_B_rate:.4f}")
    print(f"  Ratio B/A: {pre_B_rate/pre_A_rate:.2f}x\n")

    # --- 2. Effort-training log: early (right after weight transfer) vs late ---
    eff_df = pd.read_pickle(EFFORT_LOG_PATH)
    n = len(eff_df)
    early = eff_df[eff_df['step'] < 20000]
    late = eff_df[eff_df['step'] > eff_df['step'].max() - 20000]

    early_A, early_B = early['A_pull'].mean(), early['B_pull'].mean()
    late_A, late_B = late['A_pull'].mean(), late['B_pull'].mean()

    print(f"EFFORT TRAINING -- EARLY (first 20k steps, right after weight transfer):")
    print(f"  Agent A pull rate: {early_A:.4f}, Agent B pull rate: {early_B:.4f}, Ratio B/A: {early_B/early_A:.2f}x\n")
    print(f"EFFORT TRAINING -- LATE (last 20k steps):")
    print(f"  Agent A pull rate: {late_A:.4f}, Agent B pull rate: {late_B:.4f}, Ratio B/A: {late_B/late_A:.2f}x\n")

    # --- 3. Plot: pre-effort vs early vs late, side by side ---
    labels = ['Pre-effort\n(gaze generalist)', 'Effort: early\n(first 20k steps)', 'Effort: late\n(last 20k steps)']
    A_vals = [pre_A_rate, early_A, late_A]
    B_vals = [pre_B_rate, early_B, late_B]

    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.bar(x - width/2, A_vals, width, label='Agent A')
    ax.bar(x + width/2, B_vals, width, label='Agent B')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel("P(pull attempt | step)")
    ax.set_title("A vs B pull-rate imbalance: pre-existing or emergent during effort fine-tune?")
    ax.legend()

    plt.tight_layout()
    plt.savefig("pull_rate_imbalance_check.pdf", dpi=150)
    print("Saved pull_rate_imbalance_check.pdf")

    print("\n--- Interpretation guide ---")
    print("If pre-effort ratio ~= early-effort ratio: the imbalance is INHERITED from")
    print("  the gaze-cooperation training, not something the effort task introduced.")
    print("If early-effort ratio is much smaller than late-effort ratio: the imbalance")
    print("  GREW during effort fine-tuning itself -- worth checking LR / reward scale.")

In [ ]:
"""
Effort-inequity training log check.
Run on HPC in the folder containing effort_training_log.pkl.
"""
if 0:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    df = pd.read_pickle("effort_training_log.pkl")
    print(f"Loaded {len(df)} steps.")
    print(df.columns.tolist())

    # --- Reconstruct force/role info from the contingency label ---
    # cond_label format: "fixed{A/B}_A{force}g_B{force}g"
    df['fixed_agent'] = df['contingency'].str.extract(r'fixed([AB])')
    df['force_A'] = df['contingency'].str.extract(r'A(\d+)g').astype(float)
    df['force_B'] = df['contingency'].str.extract(r'B(\d+)g').astype(float)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # --- 1. Attempt rate vs own cost, for each agent, split by whether it's the varying one ---
    for agent, force_col, pull_col, ax in [('A', 'force_A', 'A_pull', axes[0,0]),
                                             ('B', 'force_B', 'B_pull', axes[0,1])]:
        is_varying = df['fixed_agent'] != agent  # varying agent = NOT the fixed one
        varying_df = df[is_varying].copy()
        varying_df['force_bin'] = (varying_df[force_col] // 100) * 100
        attempt_rate = varying_df.groupby('force_bin')[pull_col].mean()
        ax.plot(attempt_rate.index, attempt_rate.values, 'o-')
        ax.set_title(f"Agent {agent}: attempt rate vs own cost (when varying)")
        ax.set_xlabel("Force (g)")
        ax.set_ylabel("P(pull attempt | step)")

    # --- 2. Reward gap (high-cost agent minus low-cost agent) across training ---
    df['step_bin'] = (df['step'] // 20000) * 20000
    gap_when_A_varies = df[df['fixed_agent'] == 'B'].groupby('step_bin').apply(
        lambda d: d['A_reward'].sum() - d['B_reward'].sum())
    gap_when_B_varies = df[df['fixed_agent'] == 'A'].groupby('step_bin').apply(
        lambda d: d['B_reward'].sum() - d['A_reward'].sum())

    axes[1,0].plot(gap_when_A_varies.index, gap_when_A_varies.values, label='A varying (A - B)')
    axes[1,0].plot(gap_when_B_varies.index, gap_when_B_varies.values, label='B varying (B - A)')
    axes[1,0].axhline(0, color='gray', linewidth=0.8)
    axes[1,0].set_title("High-cost agent's reward deficit over training")
    axes[1,0].set_xlabel("Training step")
    axes[1,0].legend()

    # --- 3. Entropy proxy: pull-action variability per agent over training ---
    # (crude proxy since raw entropy isn't logged per-step: rolling std of pull actions)
    roll_A = df.groupby('step_bin')['A_pull'].std()
    roll_B = df.groupby('step_bin')['B_pull'].std()
    axes[1,1].plot(roll_A.index, roll_A.values, label='Agent A pull-action std')
    axes[1,1].plot(roll_B.index, roll_B.values, label='Agent B pull-action std')
    axes[1,1].set_title("Behavioral variability over training (proxy for entropy)")
    axes[1,1].set_xlabel("Training step")
    axes[1,1].legend()

    plt.tight_layout()
    plt.savefig("effort_check.pdf", dpi=150)
    print("Saved effort_check.pdf")

    # --- Print the key summary numbers ---
    print("\n--- Attempt rate by own cost (Agent A, when varying) ---")
    print(df[df['fixed_agent']=='B'].groupby((df['force_A']//100)*100)['A_pull'].mean())
    print("\n--- Attempt rate by own cost (Agent B, when varying) ---")
    print(df[df['fixed_agent']=='A'].groupby((df['force_B']//100)*100)['B_pull'].mean())